In [ ]:
import random
from rdkit import Chem
from molpher.core import MolpherMol, MolpherAtom
from molpher.core.morphing.operators import MorphingOperator
from rdkit.Chem.EnumerateStereoisomers import EnumerateStereoisomers, StereoEnumerationOptions
from rdkit.Chem import rdChemReactions
from rdkit.Chem import rdmolops
from rdkit.Chem import Descriptors  
from molpher.core import ExplorationTree as ETree


class OxidizeAlcohol(MorphingOperator):
    def __init__(self):
        super(OxidizeAlcohol, self).__init__()
        self._name = "Oxidize Alcohol"
        self._target_atoms = [] 
        self.PATTERN = Chem.MolFromSmarts("[OX2H][#6X4;H1,H2]")

    def setOriginal(self, mol):
        super(OxidizeAlcohol, self).setOriginal(mol)
        self._target_atoms = []
        
        if not self.original:
            return
            
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: 
            return
            
        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
        
            self._target_atoms.append((match[0], match[1]))

    def morph(self):
        if not self.original: 
            return None
            
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: 
            return None
            
        if not self._target_atoms:
            return MolpherMol(other=rdkit_mol)
            
        idx_o, idx_c = random.choice(self._target_atoms)
        
        try:
            rw_mol = Chem.RWMol(rdkit_mol)
            
            if rw_mol.GetBondBetweenAtoms(idx_o, idx_c):
                rw_mol.RemoveBond(idx_o, idx_c)
            rw_mol.AddBond(idx_o, idx_c, Chem.BondType.DOUBLE)
            
            new_mol = rw_mol.GetMol()
    
            for idx in [idx_o, idx_c]:
                atom = new_mol.GetAtomWithIdx(idx)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
            
        except Exception as e:
            return MolpherMol(other=rdkit_mol)

    def getName(self):
        return self._name


class OxidizeAldehydeToAcid(MorphingOperator):
    def __init__(self):
        super(OxidizeAldehydeToAcid, self).__init__()
        self._name = "Oxidize Aldehyde to Acid"
        self._target_carbons = [] 
        self.PATTERN = Chem.MolFromSmarts("[CX3H1](=O)[#6,#1]")

    def setOriginal(self, mol):
        super(OxidizeAldehydeToAcid, self).setOriginal(mol)
        self._target_carbons = []
        
        if not self.original:
            return
            
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: 
            return
            
        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            self._target_carbons.append(match[0])

    def morph(self):
        if not self.original: 
            return None
            
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: 
            return None
        
        if not self._target_carbons:
            return MolpherMol(other=rdkit_mol)
            
        idx_c = random.choice(self._target_carbons)
        
        try:
            rw_mol = Chem.RWMol(rdkit_mol)
            
            new_o_idx = rw_mol.AddAtom(Chem.Atom(8))
            
            rw_mol.AddBond(idx_c, new_o_idx, Chem.BondType.SINGLE)
            
            new_mol = rw_mol.GetMol()
            
            for idx in [idx_c, new_o_idx]:
                atom = new_mol.GetAtomWithIdx(idx)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
        
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
            
        except Exception as e:
            return MolpherMol(other=rdkit_mol)
        
    def getName(self):
        return self._name


class AlkeneToAlcohol(MorphingOperator):
    def __init__(self):
        super(AlkeneToAlcohol, self).__init__()
        self._name = "Markovnikov Hydration"
        self._target_bonds = [] 
        self.PATTERN = Chem.MolFromSmarts("[CX3;H1,H2]=[CX3;H0,H1,H2]")

    def setOriginal(self, mol):
        super(AlkeneToAlcohol, self).setOriginal(mol)
        self._target_bonds = []
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return
        
        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            bond = rdkit_mol.GetBondBetweenAtoms(match[0], match[1])
            if bond and not bond.GetIsAromatic():
                self._target_bonds.append((match[0], match[1]))

    def morph(self):
        if not self.original: return None
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return None
        if not self._target_bonds: return MolpherMol(other=rdkit_mol)
            
        idx1, idx2 = random.choice(self._target_bonds)
        
        try:
            rw_mol = Chem.RWMol(rdkit_mol)
            
            bond = rw_mol.GetBondBetweenAtoms(idx1, idx2)
            if bond:
                bond.SetBondType(Chem.BondType.SINGLE)
            
            h1 = rw_mol.GetAtomWithIdx(idx1).GetTotalNumHs()
            h2 = rw_mol.GetAtomWithIdx(idx2).GetTotalNumHs()
            idx_with_oh = idx1 if h1 <= h2 else idx2
            
            oh_idx = rw_mol.AddAtom(Chem.Atom(8))
            rw_mol.AddBond(idx_with_oh, oh_idx, Chem.BondType.SINGLE)
            
            new_mol = rw_mol.GetMol()
            for idx in [idx1, idx2, oh_idx]:
                atom = new_mol.GetAtomWithIdx(idx)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            return MolpherMol(other=new_mol)
        except:
            return MolpherMol(other=rdkit_mol)
    
    def getName(self): return self._name


class AlcoholToAlkene(MorphingOperator):
    def __init__(self):
        super(AlcoholToAlkene, self).__init__()
        self._name = "Saytzeff Dehydration"
        self._target_groups = [] 
        self.PATTERN = Chem.MolFromSmarts("[OX2H][#6X4;H1,H2;!$(C(O)=O)]")

    def setOriginal(self, mol):
        super(AlcoholToAlkene, self).setOriginal(mol)
        self._target_groups = []
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return
        
        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            self._target_groups.append((match[0], match[1]))

    def morph(self):
        if not self.original: return None
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return None
        if not self._target_groups: return MolpherMol(other=rdkit_mol)
            
        oh_idx, alpha_idx = random.choice(self._target_groups)
        alpha_atom = rdkit_mol.GetAtomWithIdx(alpha_idx)
        
        beta_carbons = [a for a in alpha_atom.GetNeighbors() if a.GetAtomicNum() == 6 and a.GetHybridization() == Chem.HybridizationType.SP3]
        if not beta_carbons: return MolpherMol(other=rdkit_mol)

        beta_carbons.sort(key=lambda x: x.GetTotalNumHs())
        beta_idx = beta_carbons[0].GetIdx()
        
        try:
            rw_mol = Chem.RWMol(rdkit_mol)
            
            bond_ab = rw_mol.GetBondBetweenAtoms(alpha_idx, beta_idx)
            if bond_ab:
                bond_ab.SetBondType(Chem.BondType.DOUBLE)
                
            bond_oh = rw_mol.GetBondBetweenAtoms(oh_idx, alpha_idx)
            if bond_oh:
                rw_mol.RemoveBond(oh_idx, alpha_idx)
            
            new_mol = rw_mol.GetMol()
            
            for idx in [alpha_idx, beta_idx]:
                atom = new_mol.GetAtomWithIdx(idx)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
            
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            
            frags = Chem.GetMolFrags(new_mol, asMols=True)
            if frags:

                frags = sorted(frags, key=lambda x: x.GetNumAtoms(), reverse=True)
                final_mol = frags[0]
                
                clean_smiles = Chem.MolToSmiles(final_mol)
                return MolpherMol(clean_smiles)
                
            return MolpherMol(other=rdkit_mol)
        except:
            return MolpherMol(other=rdkit_mol)
    
    def getName(self): return self._name


class HeteroatomOxidation(MorphingOperator):
    def __init__(self):
        super(HeteroatomOxidation, self).__init__()
        self._name = "Heteroatom Oxidation (Phase I)"
        self._matches = []
        self.N_PATTERN = Chem.MolFromSmarts("[N;X3;H0;!$(N-C=O);!a](-[#6])-[#6]")
        self.S_THIOETHER = Chem.MolFromSmarts("[S;X2;H0;!a]")
        self.S_SULFOXIDE = Chem.MolFromSmarts("[S;X3;D3;H0](=O)")

    def setOriginal(self, mol):
        super(HeteroatomOxidation, self).setOriginal(mol)
        self._matches = []
        
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return
        
        if self.N_PATTERN is not None:
            for match in rdkit_mol.GetSubstructMatches(self.N_PATTERN):
                self._matches.append((match[0], "N"))

        if self.S_THIOETHER is not None:
            for match in rdkit_mol.GetSubstructMatches(self.S_THIOETHER):
                self._matches.append((match[0], "S_thio"))

        if self.S_SULFOXIDE is not None:
            for match in rdkit_mol.GetSubstructMatches(self.S_SULFOXIDE):
                self._matches.append((match[0], "S_sulf"))

    def morph(self):
        if not self.original: return None
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return None

        if not self._matches:
            return MolpherMol(other=rdkit_mol)

        target_idx, atom_type = random.choice(self._matches)

        try:
            rw_mol = Chem.RWMol(rdkit_mol)
            
            oxygen_idx = rw_mol.AddAtom(Chem.Atom(8))
            
            target_atom = rw_mol.GetAtomWithIdx(target_idx)
            ox_atom = rw_mol.GetAtomWithIdx(oxygen_idx)

            if atom_type in ["S_thio", "S_sulf"]:

                rw_mol.AddBond(target_idx, oxygen_idx, Chem.BondType.DOUBLE)
                
                target_atom.SetNoImplicit(False)
                target_atom.SetNumExplicitHs(0)
                
            elif atom_type == "N":
                
                rw_mol.AddBond(target_idx, oxygen_idx, Chem.BondType.SINGLE)
                target_atom.SetFormalCharge(1)
                ox_atom.SetFormalCharge(-1)
                ox_atom.SetNoImplicit(True)
                ox_atom.SetNumExplicitHs(0)
                target_atom.SetNoImplicit(False)
                target_atom.SetNumExplicitHs(0)

            new_mol = rw_mol.GetMol()
            
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self):
        return self._name


class HydrolyzeEster(MorphingOperator):
    def __init__(self):
        super(HydrolyzeEster, self).__init__()
        self._name = "Ester Hydrolysis (Generalized)"
        self._matches = []
        self.PATTERN = Chem.MolFromSmarts("[CX3](=O)[OX2][#6]")

    def setOriginal(self, mol):
        super(HydrolyzeEster, self).setOriginal(mol)
        self._matches = []

        if not self.original:
            return

        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None:
            return

        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)

        for match in matches:
            carbonyl_c_idx = match[0]   
            carbonyl_o_idx = match[1]  
            ester_o_idx = match[2]      
            alkoxy_c_idx = match[3]   

            carbonyl_c = rdkit_mol.GetAtomWithIdx(carbonyl_c_idx)
            alkoxy_atom = rdkit_mol.GetAtomWithIdx(alkoxy_c_idx)
            
            oxygen_neighbors = [
                nb for nb in carbonyl_c.GetNeighbors()
                if nb.GetAtomicNum() == 8
            ]
            if len(oxygen_neighbors) > 2:
                continue

            # EXCLUSION: tert-butyl esters
            
            carbon_neighbors = [
                nb for nb in alkoxy_atom.GetNeighbors()
                if nb.GetAtomicNum() == 6
            ]
            if len(carbon_neighbors) == 3:
                continue

            self._matches.append(
                (carbonyl_c_idx, carbonyl_o_idx, ester_o_idx, alkoxy_c_idx)
            )

    def morph(self):
        if not self.original:
            return None

        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None:
            return None

        if not self._matches:
            
            return MolpherMol(other=rdkit_mol)

        carbonyl_c_idx, carbonyl_o_idx, ester_o_idx, alkoxy_c_idx = random.choice(self._matches)

        try:
            rw_mol = Chem.RWMol(rdkit_mol)

            if rw_mol.GetBondBetweenAtoms(carbonyl_c_idx, ester_o_idx):
                rw_mol.RemoveBond(carbonyl_c_idx, ester_o_idx)
            else:
                return MolpherMol(other=rdkit_mol)

            new_oh_idx = rw_mol.AddAtom(Chem.Atom(8))
            rw_mol.AddBond(carbonyl_c_idx, new_oh_idx, Chem.BondType.SINGLE)
            
            new_mol = rw_mol.GetMol()

            fragments = Chem.GetMolFrags(new_mol, asMols=True, sanitizeFrags=False)
            if not fragments:
                return MolpherMol(other=rdkit_mol)

            processed_frags = []
            for frag in fragments:
                frag_rw = Chem.RWMol(frag)
                for atom in frag_rw.GetAtoms():
                    atom.SetNoImplicit(False)
                    atom.SetNumExplicitHs(0)

                frag_mol = frag_rw.GetMol()
                try:
                    frag_mol.UpdatePropertyCache(strict=False)
                    Chem.SanitizeMol(frag_mol)
                    processed_frags.append(frag_mol)
                except Exception:
                    continue

            if not processed_frags:
                return MolpherMol(other=rdkit_mol)

            benzene = Chem.MolFromSmarts("c1ccccc1")
            ring_fragments = [f for f in processed_frags if f.HasSubstructMatch(benzene)]

            if ring_fragments:
                largest_frag = max(ring_fragments, key=lambda m: Descriptors.MolWt(m))
            else:
                largest_frag = max(processed_frags, key=lambda m: Descriptors.MolWt(m))

            largest_frag.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(largest_frag, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(largest_frag, cleanIt=True, force=True)

            return MolpherMol(other=largest_frag)

        except Exception as e:
            print(f"[Debug Error]: {e}")
            return MolpherMol(other=rdkit_mol)

    def getName(self):
        return self._name


class NDealkylation(MorphingOperator):
    def __init__(self):
        super(NDealkylation, self).__init__()
        self._name = "N-Dealkylation (Phase I - Advanced)"
        self._target_bonds = []
        self.PATTERN = Chem.MolFromSmarts("[NX3;H0,H1;!a;!$(N-C=O)][CX4;H1,H2,H3]")

    def setOriginal(self, mol):
        super(NDealkylation, self).setOriginal(mol)
        self._target_bonds = []
        
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return
        
        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            self._target_bonds.append((match[0], match[1]))

    def morph(self):
        if not self.original: return None
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return None
        
        if not self._target_bonds:
            return MolpherMol(other=rdkit_mol)
            
        idx_n, idx_c = random.choice(self._target_bonds)
        
        try:
            rw_mol = Chem.RWMol(rdkit_mol)
            
            bond = rw_mol.GetBondBetweenAtoms(idx_n, idx_c)
            if bond:
                rw_mol.RemoveBond(idx_n, idx_c)
                
            new_mol = rw_mol.GetMol()
            
            atom_n = new_mol.GetAtomWithIdx(idx_n)
            atom_n.SetNoImplicit(False)
            atom_n.SetNumExplicitHs(0)
            
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            
            frags_mols = Chem.GetMolFrags(new_mol, asMols=True)
            
            if frags_mols:
                
                frags_indices = Chem.GetMolFrags(new_mol, asMols=False)
                frags_with_meta = list(zip(frags_mols, frags_indices))
                frags_with_meta = sorted(
                    frags_with_meta, 
                    key=lambda x: (idx_n in x[1], x[0].GetNumAtoms()), 
                    reverse=True
                )
                
                final_mol = frags_with_meta[0][0]
                
                Chem.AssignStereochemistry(final_mol, cleanIt=True, force=True)
                clean_smiles = Chem.MolToSmiles(final_mol)
                return MolpherMol(clean_smiles)
                
            return MolpherMol(other=rdkit_mol)
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name


class AromaticHydroxylation(MorphingOperator):
    def __init__(self):
        super(AromaticHydroxylation, self).__init__()
        self._name = "Aromatic Hydroxylation (Phase I - Regioselective)"
        self._matches = []
        self.AROMATIC_C = Chem.MolFromSmarts("[c;H1]")

    def _get_para_score(self, mol, c_idx):
        """ Υπολογισμός para-θέσης σε 6μελείς δακτυλίους """
        for ring in mol.GetRingInfo().AtomRings():
            if c_idx in ring and len(ring) == 6:
                for r_idx in ring:
                    r_atom = mol.GetAtomWithIdx(r_idx)
                    has_ex_neighbor = any(n.GetIdx() not in ring for n in r_atom.GetNeighbors())
                    
                    if has_ex_neighbor:
                        path = Chem.GetShortestPath(mol, r_idx, c_idx)
                        if len(path) == 4: # Απόσταση para
                            return 10
        return 1 

    def setOriginal(self, mol):
        super(AromaticHydroxylation, self).setOriginal(mol)
        self._matches = []
        
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return

        if self.AROMATIC_C is not None:
            matches = rdkit_mol.GetSubstructMatches(self.AROMATIC_C)
            scored_sites = []
            for match in matches:
                c_idx = match[0]
                score = self._get_para_score(rdkit_mol, c_idx)
                scored_sites.append((c_idx, score))
                
            if scored_sites:
                max_score = max(site[1] for site in scored_sites)
                self._matches = [site[0] for site in scored_sites if site[1] == max_score]

    def morph(self):
        if not self.original: return None
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return None

        if not self._matches:
            return MolpherMol(other=rdkit_mol)

        c_idx = random.choice(self._matches)
        
        try:
            rw_mol = Chem.RWMol(rdkit_mol)
            
            new_o_idx = rw_mol.AddAtom(Chem.Atom(8))
            c_atom = rw_mol.GetAtomWithIdx(c_idx)
            o_atom = rw_mol.GetAtomWithIdx(new_o_idx)
            
            c_atom.SetNoImplicit(False)
            c_atom.SetNumExplicitHs(0)
            o_atom.SetNoImplicit(False)
            o_atom.SetNumExplicitHs(0)
            
            rw_mol.AddBond(c_idx, new_o_idx, Chem.BondType.SINGLE)
            
            new_mol = rw_mol.GetMol()
            
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name

    
class AliphaticEpoxidation(MorphingOperator):
    def __init__(self):
        super(AliphaticEpoxidation, self).__init__()
        self._name = "Aliphatic Epoxidation (Phase I - Stereospecific)"
        self._target_bonds = []
        self.DOUBLE_BOND_PATTERN = Chem.MolFromSmarts("[CX3;!a]=[CX3;!a]")

    def setOriginal(self, mol):
        super(AliphaticEpoxidation, self).setOriginal(mol)
        self._target_bonds = []
        
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return

        if self.DOUBLE_BOND_PATTERN is not None:
            matches = rdkit_mol.GetSubstructMatches(self.DOUBLE_BOND_PATTERN)
            for match in matches:
                
                pair = tuple(sorted([match[0], match[1]]))
                if pair not in self._target_bonds:
                    self._target_bonds.append(pair)

    def morph(self):
        if not self.original: return None
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return None

        if not self._target_bonds:
            return MolpherMol(other=rdkit_mol)
        
        idx1, idx2 = random.choice(self._target_bonds)

        try:
            rw_mol = Chem.RWMol(rdkit_mol)
            
            bond = rw_mol.GetBondBetweenAtoms(idx1, idx2)
            if bond:
                bond.SetBondType(Chem.BondType.SINGLE)
            
            oxygen_idx = rw_mol.AddAtom(Chem.Atom(8))
            
            rw_mol.AddBond(idx1, oxygen_idx, Chem.BondType.SINGLE)
            rw_mol.AddBond(idx2, oxygen_idx, Chem.BondType.SINGLE)
            
            new_mol = rw_mol.GetMol()

            for idx in [idx1, idx2, oxygen_idx]:
                atom = new_mol.GetAtomWithIdx(idx)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
    
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self):
        return self._name


class AzoReduction(MorphingOperator):
    def __init__(self):
        super(AzoReduction, self).__init__()
        self._name = "Azo Reduction (Phase I - Safe Fragmentation)"
        self._matches = []
        self.AZO_PATTERN = Chem.MolFromSmarts("[C,c]-[N;!R]=[N;!R]-[C,c]")

    def setOriginal(self, mol):
        super(AzoReduction, self).setOriginal(mol)
        self._matches = []

        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return

        matches = rdkit_mol.GetSubstructMatches(self.AZO_PATTERN)
        for match in matches:
            self._matches.append((match[0], match[1], match[2], match[3]))

    def morph(self):
        if not self.original: return None
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return None

        if not self._matches:
            return MolpherMol(other=rdkit_mol)

        edit_mol = Chem.RWMol(rdkit_mol)
        c1_idx, n1_idx, n2_idx, c2_idx = random.choice(self._matches)

        try:
            edit_mol.RemoveBond(n1_idx, n2_idx)

            for idx in [n1_idx, n2_idx]:
                atom = edit_mol.GetAtomWithIdx(idx)
                atom.SetFormalCharge(0)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                for prop in list(atom.GetPropNames()):
                    atom.ClearProp(prop)

            new_mol = edit_mol.GetMol()
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            
            fragments = rdmolops.GetMolFrags(new_mol, asMols=True)
            if not fragments:
                return MolpherMol(other=rdkit_mol)

            chosen_frag = random.choice(fragments)
            
            Chem.AssignStereochemistry(chosen_frag, cleanIt=True, force=True)
            return MolpherMol(other=chosen_frag)
                
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name

    
class NitroReduction(MorphingOperator):
    def __init__(self):
        super(NitroReduction, self).__init__()
        self._name = "Nitro Reduction (Phase I - Safe)"
        self._target_nitrogens = []
        self.NITRO_PATTERN = Chem.MolFromSmarts("[C,c][N;X3](=[O,O-])~[O,O-]")

    def setOriginal(self, mol):
        super(NitroReduction, self).setOriginal(mol)
        self._target_nitrogens = []

        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return

        matches = rdkit_mol.GetSubstructMatches(self.NITRO_PATTERN)
        for match in matches:
            if match[1] not in self._target_nitrogens:
                self._target_nitrogens.append(match[1])

    def morph(self):
        if not self.original: return None
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return None

        if not self._target_nitrogens:
            return MolpherMol(other=rdkit_mol)

        n_idx = random.choice(self._target_nitrogens)

        try:
            rw_mol = Chem.RWMol(rdkit_mol)
            n_atom = rw_mol.GetAtomWithIdx(n_idx)
            
            o_indices = [neighbor.GetIdx() for neighbor in n_atom.GetNeighbors() if neighbor.GetAtomicNum() == 8]
            
            for o_idx in o_indices:
                bond = rw_mol.GetBondBetweenAtoms(n_idx, o_idx)
                if bond:
                    rw_mol.RemoveBond(n_idx, o_idx)
            
            n_atom.SetFormalCharge(0)
            n_atom.SetNoImplicit(False)
            n_atom.SetNumExplicitHs(2)
            for prop in list(n_atom.GetPropNames()):
                n_atom.ClearProp(prop)
            
            new_mol = rw_mol.GetMol()
            new_mol.UpdatePropertyCache(strict=False)
            
            atoms_to_remove = []
            for atom in new_mol.GetAtoms():
                if atom.GetAtomicNum() == 8 and atom.GetDegree() == 0:
                    atoms_to_remove.append(atom.GetIdx())

            edit = Chem.RWMol(new_mol)
            for idx in sorted(atoms_to_remove, reverse=True):
                edit.RemoveAtom(idx)
            new_mol = edit.GetMol()

            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            return MolpherMol(other=new_mol)

        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name


class AldehydeReduction(MorphingOperator):
    def __init__(self):
        super(AldehydeReduction, self).__init__()
        self._name = "Aldehyde Reduction (Phase I - Safe)"
        self._matches = []
        self.ALDEHYDE_PATTERN = Chem.MolFromSmarts("[CX3H1;!$(C(=O)[O,N,S,F,Cl,Br,I])]=[OX1]")

    def setOriginal(self, mol):
        super(AldehydeReduction, self).setOriginal(mol)
        self._matches = []

        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return

        matches = rdkit_mol.GetSubstructMatches(self.ALDEHYDE_PATTERN)
        for match in matches:
            self._matches.append((match[0], match[1]))

    def morph(self):
        if not self.original: return None
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return None

        if not self._matches:
            return MolpherMol(other=rdkit_mol)

        edit_mol = Chem.RWMol(rdkit_mol)
        c_idx, o_idx = random.choice(self._matches)

        try:
            bond = edit_mol.GetBondBetweenAtoms(c_idx, o_idx)
            if bond is None: return MolpherMol(other=rdkit_mol)

            bond.SetBondType(Chem.BondType.SINGLE)

            for idx in [c_idx, o_idx]:
                atom = edit_mol.GetAtomWithIdx(idx)
                atom.SetFormalCharge(0)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                for prop in list(atom.GetPropNames()):
                    atom.ClearProp(prop)
                atom.UpdatePropertyCache(strict=False)

            new_mol = edit_mol.GetMol()
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name


class KetoneReduction(MorphingOperator):
    def __init__(self):
        super(KetoneReduction, self).__init__()
        self._name = "Ketone Reduction (Phase I - Stereospecific Enumerate)"
        self._matches = []

    def setOriginal(self, mol):
        super(KetoneReduction, self).setOriginal(mol)
        self._matches = []

        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return
        
        ketone_pattern = Chem.MolFromSmarts("[CX3;$(C(=O)(-[#6])-[#6])]=O")
        matches = rdkit_mol.GetSubstructMatches(ketone_pattern)
        for match in matches:
            self._matches.append((match[0], match[1]))

    def morph(self):
        if not self.original: return None
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return None

        if not self._matches:
            return MolpherMol(other=rdkit_mol)

        edit_mol = Chem.RWMol(rdkit_mol)
        carbonyl_idx, oxygen_idx = random.choice(self._matches)

        try:
            bond = edit_mol.GetBondBetweenAtoms(carbonyl_idx, oxygen_idx)
            if bond is None: return MolpherMol(other=rdkit_mol)

            bond.SetBondType(Chem.BondType.SINGLE)

            for idx in [carbonyl_idx, oxygen_idx]:
                atom = edit_mol.GetAtomWithIdx(idx)
                atom.SetFormalCharge(0)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                atom.SetChiralTag(Chem.ChiralType.CHI_UNSPECIFIED)
                if atom.HasProp('_CIPCode'): atom.ClearProp('_CIPCode')
                for prop in list(atom.GetPropNames()): atom.ClearProp(prop)
                atom.UpdatePropertyCache(strict=False)

            new_mol = edit_mol.GetMol()
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)

            options = StereoEnumerationOptions(onlyUnassigned=True)
            isomers = list(EnumerateStereoisomers(new_mol, options=options))

            if isomers:
                chosen_iso = random.choice(isomers)
                Chem.SanitizeMol(chosen_iso)
                Chem.AssignStereochemistry(chosen_iso, cleanIt=True, force=True)
                return MolpherMol(other=chosen_iso)

            return MolpherMol(other=new_mol)
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name


class AlkenylReduction(MorphingOperator):
    def __init__(self):
        super(AlkenylReduction, self).__init__()
        self._name = "Alkenyl Reduction (Phase I - Ring Safe)"
        self._target_bonds = []
        self.PATTERN = Chem.MolFromSmarts("[CX3;!a]=[CX3;!a]")

    def setOriginal(self, mol):
        super(AlkenylReduction, self).setOriginal(mol)
        self._target_bonds = []
        
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return

        if self.PATTERN is not None:
            matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
            for match in matches:
                pair = tuple(sorted([match[0], match[1]]))
                if pair not in self._target_bonds:
                    self._target_bonds.append(pair)

    def morph(self):
        if not self.original: return None
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return None

        if not self._target_bonds:
            return MolpherMol(other=rdkit_mol)
        
        idx1, idx2 = random.choice(self._target_bonds)

        try:
            rw_mol = Chem.RWMol(rdkit_mol)
            
            bond = rw_mol.GetBondBetweenAtoms(idx1, idx2)
            if bond:
                bond.SetBondType(Chem.BondType.SINGLE)
                bond.SetStereo(Chem.BondStereo.STEREONONE)
            
            for idx in [idx1, idx2]:
                atom = rw_mol.GetAtomWithIdx(idx)
                atom.SetFormalCharge(0)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                atom.SetChiralTag(Chem.ChiralType.CHI_UNSPECIFIED)
                if atom.HasProp('_CIPCode'): 
                    atom.ClearProp('_CIPCode')
                for prop in list(atom.GetPropNames()):
                    atom.ClearProp(prop)
                atom.UpdatePropertyCache(strict=False)
            
            new_mol = rw_mol.GetMol()
            
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self):
        return self._name


class LactamFormation(MorphingOperator):
    def __init__(self):
        super(LactamFormation, self).__init__()
        self._name = "Lactam Formation (Phase I - Ring Oxidation Safe)"
        self._matches = []
        self.PATTERN = Chem.MolFromSmarts("[NX3;R;!$(N-C=O);!a]-[CX4;R;H2;!$(C-O)]")

    def setOriginal(self, mol):
        super(LactamFormation, self).setOriginal(mol)
        self._matches = []

        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return

        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            self._matches.append((match[0], match[1]))

    def morph(self):
        if not self.original: return None
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return None

        if not self._matches:
            return MolpherMol(other=rdkit_mol)

        n_idx, c_idx = random.choice(self._matches)

        try:
            rw_mol = Chem.RWMol(rdkit_mol)
            
            o_atom = Chem.Atom(8)
            o_atom.SetFormalCharge(0)
            o_idx = rw_mol.AddAtom(o_atom)
            
            rw_mol.AddBond(c_idx, o_idx, Chem.BondType.DOUBLE)
            
            for idx in [n_idx, c_idx, o_idx]:
                atom = rw_mol.GetAtomWithIdx(idx)
                atom.SetFormalCharge(0)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                for prop in list(atom.GetPropNames()):
                    atom.ClearProp(prop)
                atom.UpdatePropertyCache(strict=False)
            
            new_mol = rw_mol.GetMol()
            
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self):
        return self._name


class AcylGlucuronidation(MorphingOperator):
    def __init__(self):
        super(AcylGlucuronidation, self).__init__()
        self._name = "Acyl Glucuronidation (Carboxylic Acids)"
        self._target_oxygens = []
        self.PATTERN = Chem.MolFromSmarts("[CX3](=O)[OX2H]")
        self.GLUCURONIDE_TEMPLATE = Chem.MolFromSmiles("C1(O)O[C@H](C(=O)O)[C@@H](O)[C@H](O)[C@@H]1O")

    def setOriginal(self, mol):
        super(AcylGlucuronidation, self).setOriginal(mol)
        self._target_oxygens = []
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return
        
        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            self._target_oxygens.append(match[2])

    def morph(self):
        if not self.original or not self._target_oxygens:
            return MolpherMol(other=self.original.asRDMol())
        
        rdkit_mol = self.original.asRDMol()
        try:
            target_o_idx = random.choice(self._target_oxygens)
            
            combined = Chem.CombineMols(rdkit_mol, self.GLUCURONIDE_TEMPLATE)
            rw_combined = Chem.RWMol(combined)
            
            c_sugar_idx = rdkit_mol.GetNumAtoms()
            oh_sugar_idx = c_sugar_idx + 1
            
            rw_combined.AddBond(target_o_idx, c_sugar_idx, Chem.BondType.SINGLE)
        
            rw_combined.RemoveAtom(oh_sugar_idx)
        
            new_mol = rw_combined.GetMol()
            
            for idx in [target_o_idx, c_sugar_idx]:
                atom = new_mol.GetAtomWithIdx(idx)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
            
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name

   
class AlcoholPhenolGlucuronidation(MorphingOperator):
    def __init__(self):
        super(AlcoholPhenolGlucuronidation, self).__init__()
        self._name = "O-Glucuronidation (Alcohols/Phenols)"
        self._target_oxygens = []
        
        # SMARTS: Επιλέγει [OX2H] που συνδέεται με άνθρακα, ο οποίος ΔΕΝ είναι καρβονύλιο
        self.PATTERN = Chem.MolFromSmarts("[#6;!$(C=O);!$(C=C)][OX2H]")
        # Template: β-D-glucuronide με SMILES όπου ο C1 (ανωμερής) είναι το 1ο άτομο (index 0)
        # SMILES: C1([OH])O[C@H](C(=O)O)[C@@H](O)[C@H](O)[C@@H]1O
        # Με αυτό το SMILES: 
        # index 0 -> ο C1 που θα ενωθεί με το υπόλοιπο μόριο
        # index 1 -> το -OH του C1 το οποίο ΠΡΕΠΕΙ να αφαιρεθεί
        self.GLUCURONIDE_TEMPLATE = Chem.MolFromSmiles("C1(O)O[C@H](C(=O)O)[C@@H](O)[C@H](O)[C@@H]1O")

    def setOriginal(self, mol):
        super(AlcoholPhenolGlucuronidation, self).setOriginal(mol)
        self._target_oxygens = []
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return

        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            # Το [OX2H] είναι το δεύτερο άτομο στο pattern (index 1)
            self._target_oxygens.append(match[1])

    def morph(self):
        if not self.original or not self._target_oxygens:
            return MolpherMol(other=self.original.asRDMol())

        rdkit_mol = self.original.asRDMol()
        try:
            target_o_idx = random.choice(self._target_oxygens)
            
            combined = Chem.CombineMols(rdkit_mol, self.GLUCURONIDE_TEMPLATE)
            rw_combined = Chem.RWMol(combined)
            c_sugar_idx = rdkit_mol.GetNumAtoms() 
            oh_sugar_idx = c_sugar_idx + 1 
            
            rw_combined.AddBond(target_o_idx, c_sugar_idx, Chem.BondType.SINGLE)
            rw_combined.RemoveAtom(oh_sugar_idx)
            
            new_mol = rw_combined.GetMol()
            
            for idx in [target_o_idx, c_sugar_idx]:
                atom = new_mol.GetAtomWithIdx(idx)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
            
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name


    
class AlcoholPhenolSulfation(MorphingOperator):
    def __init__(self):
        super(AlcoholPhenolSulfation, self).__init__()
        self._name = "O-Sulfation (Alcohols/Phenols)"
        self._target_atoms = []
        self.PATTERN = Chem.MolFromSmarts("[OX2H][#6;!$(C=O)]")

    def setOriginal(self, mol):
        super(AlcoholPhenolSulfation, self).setOriginal(mol)
        self._target_atoms = []

        if not self.original:
            return

        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None:
            return

        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            self._target_atoms.append(match[0])

    def morph(self):
        if not self.original:
            return None

        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None:
            return None

        if not self._target_atoms:
            return MolpherMol(other=rdkit_mol)

        oxygen_idx = random.choice(self._target_atoms)

        try:
            rw_mol = Chem.RWMol(rdkit_mol)

            # Εισαγωγή ουδέτερης ομάδας -SO3H
            sulfur_idx = rw_mol.AddAtom(Chem.Atom(16)) # S
            rw_mol.AddBond(oxygen_idx, sulfur_idx, Chem.BondType.SINGLE)

            o1_idx = rw_mol.AddAtom(Chem.Atom(8)) # =O
            rw_mol.AddBond(sulfur_idx, o1_idx, Chem.BondType.DOUBLE)

            o2_idx = rw_mol.AddAtom(Chem.Atom(8)) # =O
            rw_mol.AddBond(sulfur_idx, o2_idx, Chem.BondType.DOUBLE)

            o3_idx = rw_mol.AddAtom(Chem.Atom(8)) # -OH (Ουδέτερο)
            rw_mol.AddBond(sulfur_idx, o3_idx, Chem.BondType.SINGLE)

            new_mol = rw_mol.GetMol()

            for idx in [oxygen_idx, sulfur_idx, o1_idx, o2_idx, o3_idx]:
                atom = new_mol.GetAtomWithIdx(idx)
                if atom.GetAtomicNum() != 16: # Αφήνουμε το Θείο να διαχειριστεί το σθένος του (6)
                    atom.SetNoImplicit(False)
                    atom.SetNumExplicitHs(0)

            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)

            return MolpherMol(other=new_mol)

        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self):
        return self._name


class NitrogenSulfation(MorphingOperator):
    def __init__(self):
        super(NitrogenSulfation, self).__init__()
        self._name = "N-Sulfation (Amines 1°, 2°, 3° & Aromatic)"
        self._target_atoms = []
        self.PATTERN = Chem.MolFromSmarts("[N;X3;!$(N-C=O);!$(N-C(=O)N);!$(N-C(=O)O)]")

    def setOriginal(self, mol):
        super(NitrogenSulfation, self).setOriginal(mol)
        self._target_atoms = []
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return
        
        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            self._target_atoms.append(match[0])

    def morph(self):
        if not self.original: return None
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return None
        
        if not self._target_atoms:
            return MolpherMol(other=rdkit_mol)
        
        nitrogen_idx = random.choice(self._target_atoms)
        
        try:
            rw_mol = Chem.RWMol(rdkit_mol)
            is_tertiary = (rw_mol.GetAtomWithIdx(nitrogen_idx).GetTotalNumHs() == 0)
            
            sulfur_idx = rw_mol.AddAtom(Chem.Atom(16)) # S
            rw_mol.AddBond(nitrogen_idx, sulfur_idx, Chem.BondType.SINGLE)
            
            o1_idx = rw_mol.AddAtom(Chem.Atom(8)) # =O
            rw_mol.AddBond(sulfur_idx, o1_idx, Chem.BondType.DOUBLE)
            
            o2_idx = rw_mol.AddAtom(Chem.Atom(8)) # =O
            rw_mol.AddBond(sulfur_idx, o2_idx, Chem.BondType.DOUBLE)
            
            o3_idx = rw_mol.AddAtom(Chem.Atom(8)) # -OH
            rw_mol.AddBond(sulfur_idx, o3_idx, Chem.BondType.SINGLE)
            
            new_mol = rw_mol.GetMol()
            
            for idx in [o1_idx, o2_idx, o3_idx]:
                atom = new_mol.GetAtomWithIdx(idx)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                atom.SetFormalCharge(0)
            
            n_atom = new_mol.GetAtomWithIdx(nitrogen_idx)
            n_atom.SetNoImplicit(False)
            n_atom.SetNumExplicitHs(0)
            
            if is_tertiary:
                
                n_atom.SetFormalCharge(1)
            else:
                
                n_atom.SetFormalCharge(0)
                
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
            
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name
    

class AminoAcidConjugation(MorphingOperator):
    def __init__(self):
        super(AminoAcidConjugation, self).__init__()
        self._name = "Amino Acid Conjugation (Gly/Tau/Gln)"
        self._target_atoms = []
        self.PATTERN = Chem.MolFromSmarts("[CX3](=O)[OX2H]") 
        # Σε όλα, το Άζωτο (Ν) που θα επιτεθεί είναι το ΠΡΩΤΟ άτομο (index 0)
        self.TEMPLATES = {
            "Glycine": Chem.MolFromSmiles("NCC(=O)O"),
            "Taurine": Chem.MolFromSmiles("NCCS(=O)(=O)O"),
            "Glutamine": Chem.MolFromSmiles("N[C@@H](CCC(=O)N)C(=O)O") # Διατήρηση στερεοχημείας
        }

    def setOriginal(self, mol):
        super(AminoAcidConjugation, self).setOriginal(mol)
        self._target_atoms = []
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return
        
        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            # Κρατάμε το index του Καρβονυλικού Άνθρακα (index 0) και του -OH (index 2)
            self._target_atoms.append((match[0], match[2]))

    def morph(self):
        if not self.original or not self._target_atoms:
            return MolpherMol(other=self.original.asRDMol())
        
        rdkit_mol = self.original.asRDMol()
        try:
            chosen_c_idx, chosen_oh_idx = random.choice(self._target_atoms)
            template_name = random.choice(list(self.TEMPLATES.keys()))
            template_mol = self.TEMPLATES[template_name]
            combined = Chem.CombineMols(rdkit_mol, template_mol)
            rw_combined = Chem.RWMol(combined)
            n_amino_idx = rdkit_mol.GetNumAtoms()
            rw_combined.AddBond(chosen_c_idx, n_amino_idx, Chem.BondType.SINGLE)
            rw_combined.RemoveAtom(chosen_oh_idx)
            
            new_mol = rw_combined.GetMol()
            
            # Λόγω του RemoveAtom, ο index του n_amino_idx μετατοπίστηκε κατά -1 
            # αν ο chosen_oh_idx ήταν μικρότερος, αλλά για ασφάλεια κάνουμε reset στο στοχευμένο καρβονύλιο 
            # και στο άζωτο σαρώνοντας το γράφημα.
            for atom in new_mol.GetAtoms():
                if atom.GetAtomicNum() in [6, 7]: # Άνθρακας καρβονυλίου και Άζωτο αμιδίου
                    atom.SetNoImplicit(False)
                    atom.SetNumExplicitHs(0)
                    atom.SetFormalCharge(0)
            
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
            
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name

    
class NAcetylation(MorphingOperator):
    def __init__(self):
        super(NAcetylation, self).__init__()
        self._name = "N-acetylation (Primary Amines)"
        self._target_atoms = []
        self.PATTERN = Chem.MolFromSmarts("[N;H2;!$(N-C=O);!$(N-S(=O)=O);!$(NN)]")

    def setOriginal(self, mol):
        super(NAcetylation, self).setOriginal(mol)
        self._target_atoms = []
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return
        
        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            self._target_atoms.append(match[0])

    def morph(self):
        if not self.original or not self._target_atoms:
            return MolpherMol(other=self.original.asRDMol())
        
        rdkit_mol = self.original.asRDMol()
        try:
            chosen_idx = random.choice(self._target_atoms)
            rw_mol = Chem.RWMol(rdkit_mol)
            
            c_carbonyl_idx = rw_mol.AddAtom(Chem.Atom(6))
            rw_mol.AddBond(chosen_idx, c_carbonyl_idx, Chem.BondType.SINGLE)
            
            o_carbonyl_idx = rw_mol.AddAtom(Chem.Atom(8))
            rw_mol.AddBond(c_carbonyl_idx, o_carbonyl_idx, Chem.BondType.DOUBLE)
            
            c_methyl_idx = rw_mol.AddAtom(Chem.Atom(6))
            rw_mol.AddBond(c_carbonyl_idx, c_methyl_idx, Chem.BondType.SINGLE)
            
            new_mol = rw_mol.GetMol()
            
            for idx in [chosen_idx, c_carbonyl_idx, o_carbonyl_idx, c_methyl_idx]:
                atom = new_mol.GetAtomWithIdx(idx)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                atom.SetFormalCharge(0)
                
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
            
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name


class HydrazineAcetylation(MorphingOperator):
    def __init__(self):
        super(HydrazineAcetylation, self).__init__()
        self._name = "Hydrazine acetylation"
        self._target_atoms = []
        self.PATTERN = Chem.MolFromSmarts("[NX3;!$(N-C=O)][NX3;H2;!$(N-C=O)]")

    def setOriginal(self, mol):
        super(HydrazineAcetylation, self).setOriginal(mol)
        self._target_atoms = []
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return
        
        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            # Κρατάμε ΜΟΝΟ το index του ακραίου αζώτου [-NH2] 
            self._target_atoms.append(match[1])

    def morph(self):
        if not self.original or not self._target_atoms:
            return MolpherMol(other=self.original.asRDMol())
        
        rdkit_mol = self.original.asRDMol()
        try:
            terminal_n_idx = random.choice(self._target_atoms)
            rw_mol = Chem.RWMol(rdkit_mol)
            
            c_carbonyl_idx = rw_mol.AddAtom(Chem.Atom(6))
            rw_mol.AddBond(terminal_n_idx, c_carbonyl_idx, Chem.BondType.SINGLE)
            
            o_carbonyl_idx = rw_mol.AddAtom(Chem.Atom(8))
            rw_mol.AddBond(c_carbonyl_idx, o_carbonyl_idx, Chem.BondType.DOUBLE)
            
            c_methyl_idx = rw_mol.AddAtom(Chem.Atom(6))
            rw_mol.AddBond(c_carbonyl_idx, c_methyl_idx, Chem.BondType.SINGLE)
            
            new_mol = rw_mol.GetMol()
            
            for idx in [terminal_n_idx, c_carbonyl_idx, o_carbonyl_idx, c_methyl_idx]:
                atom = new_mol.GetAtomWithIdx(idx)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                atom.SetFormalCharge(0)
                
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
            
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name

  
class HydrazideAcetylation(MorphingOperator):
    def __init__(self):
        super(HydrazideAcetylation, self).__init__()
        self._name = "Hydrazide acetylation"
        self._target_atoms = []
        self.PATTERN = Chem.MolFromSmarts("[CX3](=O)[NX3;H1][NX3;H2;!$(N-C=O)]")

    def setOriginal(self, mol):
        super(HydrazideAcetylation, self).setOriginal(mol)
        self._target_atoms = []
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return
        
        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            # Κρατάμε το index του ακραίου αζώτου [-NH2] (index 3)
            self._target_atoms.append(match[3])

    def morph(self):
        if not self.original or not self._target_atoms:
            return MolpherMol(other=self.original.asRDMol())
        
        rdkit_mol = self.original.asRDMol()
        try:
            terminal_n_idx = random.choice(self._target_atoms)
            rw_mol = Chem.RWMol(rdkit_mol)
            
            c_carbonyl_idx = rw_mol.AddAtom(Chem.Atom(6))
            rw_mol.AddBond(terminal_n_idx, c_carbonyl_idx, Chem.BondType.SINGLE)
            
            o_carbonyl_idx = rw_mol.AddAtom(Chem.Atom(8))
            rw_mol.AddBond(c_carbonyl_idx, o_carbonyl_idx, Chem.BondType.DOUBLE)
            
            c_methyl_idx = rw_mol.AddAtom(Chem.Atom(6))
            rw_mol.AddBond(c_carbonyl_idx, c_methyl_idx, Chem.BondType.SINGLE)
            
            new_mol = rw_mol.GetMol()
 
            for idx in [terminal_n_idx, c_carbonyl_idx, o_carbonyl_idx, c_methyl_idx]:
                atom = new_mol.GetAtomWithIdx(idx)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                atom.SetFormalCharge(0)
                
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
            
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name
    

class OMethylation(MorphingOperator):
    def __init__(self):
        super(OMethylation, self).__init__()
        self._name = "O-methylation (Alcohols/Phenols)"
        self._target_atoms = []
        self.PATTERN = Chem.MolFromSmarts("[OX2H][#6;!$(C=O);!$(C=C);!$(C=N)]")

    def setOriginal(self, mol):
        super(OMethylation, self).setOriginal(mol)
        self._target_atoms = []
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return
        
        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            self._target_atoms.append(match[0])

    def morph(self):
        if not self.original or not self._target_atoms:
            return MolpherMol(other=self.original.asRDMol())
        
        rdkit_mol = self.original.asRDMol()
        try:
            target_o_idx = random.choice(self._target_atoms)
            rw_mol = Chem.RWMol(rdkit_mol)
            
            c_methyl_idx = rw_mol.AddAtom(Chem.Atom(6))
            rw_mol.AddBond(target_o_idx, c_methyl_idx, Chem.BondType.SINGLE)
            
            new_mol = rw_mol.GetMol()
            
            for idx in [target_o_idx, c_methyl_idx]:
                atom = new_mol.GetAtomWithIdx(idx)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                atom.SetFormalCharge(0)
                
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
            
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name

  
class NMethylation(MorphingOperator):
    def __init__(self):
        super(NMethylation, self).__init__()
        self._name = "N-methylation (Primary Amines)"
        self._target_atoms = []
        
        # Ενισχυμένο SMARTS Pattern για Πρωτοταγείς Αμίνες:
        # Ζητάμε άζωτο με 2 υδρογόνα [N;H2]
        # Αποκλείουμε: Αμίδια (!$(N-C=O)), Υδραζίνες (!$(NN)), 
        # Σουλφοναμίδια (!$(N-S(=O)=O)), Ουρίες/Καρβαμιδικά (!$(N-C(=O)))
        self.PATTERN = Chem.MolFromSmarts("[N;H2;!$(N-C=O);!$(N-S(=O)=O);!$(NN)]")

    def setOriginal(self, mol):
        super(NMethylation, self).setOriginal(mol)
        self._target_atoms = []
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return
        
        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            self._target_atoms.append(match[0])

    def morph(self):
        if not self.original or not self._target_atoms:
            return MolpherMol(other=self.original.asRDMol())
        
        rdkit_mol = self.original.asRDMol()
        try:
            target_n_idx = random.choice(self._target_atoms)
            rw_mol = Chem.RWMol(rdkit_mol)
            
            # Προσθήκη του Άνθρακα του Μεθυλίου (-CH3)
            c_methyl_idx = rw_mol.AddAtom(Chem.Atom(6))
            rw_mol.AddBond(target_n_idx, c_methyl_idx, Chem.BondType.SINGLE)
            
            new_mol = rw_mol.GetMol()
            
            # Reset ιδιοτήτων στα άτομα που τροποποιήθηκαν
            for idx in [target_n_idx, c_methyl_idx]:
                atom = new_mol.GetAtomWithIdx(idx)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                atom.SetFormalCharge(0)
                
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
            
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name
    

class SMethylation(MorphingOperator):
    def __init__(self):
        super(SMethylation, self).__init__()
        self._name = "S-methylation (Thiols)"
        self._target_atoms = []
        self.PATTERN = Chem.MolFromSmarts("[SX2H][#6;!$(C=O);!$(C=S)]")

    def setOriginal(self, mol):
        super(SMethylation, self).setOriginal(mol)
        self._target_atoms = []
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return
        
        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            # Κρατάμε το index του θείου [-SH] (το πρώτο άτομο του match, index 0)
            self._target_atoms.append(match[0])

    def morph(self):
        if not self.original or not self._target_atoms:
            return MolpherMol(other=self.original.asRDMol())
        
        rdkit_mol = self.original.asRDMol()
        try:
            target_s_idx = random.choice(self._target_atoms)
            rw_mol = Chem.RWMol(rdkit_mol)
            
            c_methyl_idx = rw_mol.AddAtom(Chem.Atom(6))
            rw_mol.AddBond(target_s_idx, c_methyl_idx, Chem.BondType.SINGLE)
            
            new_mol = rw_mol.GetMol()
            
            for idx in [target_s_idx, c_methyl_idx]:
                atom = new_mol.GetAtomWithIdx(idx)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                atom.SetFormalCharge(0)
                
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
            
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name

    
oxidize_alcohol_op = OxidizeAlcohol()
oxidize_aldehyde_op = OxidizeAldehydeToAcid() 
op_hydration = AlkeneToAlcohol()
op_dehydration = AlcoholToAlkene()
hetero_op = HeteroatomOxidation()
hydrolize_ester = HydrolyzeEster()
dealk_op = NDealkylation()
hydroxylation_op = AromaticHydroxylation()
epox_op = AliphaticEpoxidation()
azo_op = AzoReduction()
nitro_op = NitroReduction()
aldehyde_op = AldehydeReduction()
ketone_op = KetoneReduction()
alkene_op = AlkenylReduction()
lactam_op = LactamFormation()
acyl_glucuronidation = AcylGlucuronidation()
alph_glucoronidation = AlcoholPhenolGlucuronidation()
sulfation_op = AlcoholPhenolSulfation()  
nitrogen_sulfation = NitrogenSulfation()
amino_op = AminoAcidConjugation()
nacet_op = NAcetylation()
hydrazine_acetylation_op = HydrazineAcetylation()  
hydrazide_acetylation_op = HydrazideAcetylation()
omethylation_op = OMethylation()  
nmethylation_op = NMethylation()
smethylation_op = SMethylation()  


start_smiles = input("Δώσε SMILES για το start molecule: ")
target_smiles = input("Δώσε SMILES για το target molecule: ")
start_mol = MolpherMol(start_smiles)
target_mol = MolpherMol(target_smiles)
rdkit_start = start_mol.asRDMol()

selected_operators = []

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[OX2H][#6X4;H1,H2]")):
    selected_operators.append(oxidize_alcohol_op)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[CX3H1](=O)[#6,#1]")):
    selected_operators.append(oxidize_aldehyde_op)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[CX3;H1,H2]=[CX3;H0,H1,H2]")):
    selected_operators.append(op_hydration)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[OX2H][#6X4;H1,H2;!$(C(O)=O)]")):
    selected_operators.append(op_dehydration)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[N;X3;H0;!$(N-C=O);!a](-[#6])-[#6]")):
    selected_operators.append(hetero_op)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[S;X2;H0;!a]")):
    selected_operators.append(hetero_op)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[S;X3;D3;H0](=O)")):
    selected_operators.append(hetero_op)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[CX3](=O)[OX2][#6]")):
    selected_operators.append(hydrolize_ester)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[NX3;H0,H1;!a;!$(N-C=O)][CX4;H1,H2,H3]")):
    selected_operators.append(dealk_op)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[c;H1]")):
    selected_operators.append(hydroxylation_op)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[CX3;!a]=[CX3;!a]")):
    selected_operators.append(epox_op)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[C,c]-[N;!R]=[N;!R]-[C,c]")):
    selected_operators.append(azo_op)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[C,c][N;X3](=[O,O-])~[O,O-]")):
    selected_operators.append(nitro_op)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[CX3H1;!$(C(=O)[O,N,S,F,Cl,Br,I])]=[OX1]")):
    selected_operators.append(aldehyde_op)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[CX3;$(C(=O)(-[#6])-[#6])]=O")):
    selected_operators.append(ketone_op)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[CX3;!a]=[CX3;!a]")):
    selected_operators.append(alkene_op)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[NX3;R;!$(N-C=O);!a]-[CX4;R;H2;!$(C-O)]")):
    selected_operators.append(lactam_op)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[CX3](=O)[OX2H]")):
    selected_operators.append(acyl_glucuronidation)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[#6;!$(C=O);!$(C=C)][OX2H]")):
    selected_operators.append(alph_glucoronidation)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[OX2H][#6;!$(C=O)]")):
    selected_operators.append(sulfation_op)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[N;X3;!$(N-C=O);!$(N-C(=O)N);!$(N-C(=O)O)]")):
    selected_operators.append(nitrogen_sulfation)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[CX3](=O)[OX2H]")):
    selected_operators.append(amino_op)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[N;H2;!$(N-C=O);!$(N-S(=O)=O);!$(NN)]")):
    selected_operators.append(nacet_op)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[NX3;!$(N-C=O)][NX3;H2;!$(N-C=O)]")):
    selected_operators.append(hydrazine_acetylation_op)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[CX3](=O)[NX3;H1][NX3;H2;!$(N-C=O)]")):
    selected_operators.append(hydrazide_acetylation_op)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[OX2H][#6;!$(C=O);!$(C=C);!$(C=N)]")):
    selected_operators.append(omethylation_op)
    
if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[N;H2;!$(N-C=O);!$(N-S(=O)=O);!$(NN)]")):
    selected_operators.append(nmethylation_op)
    
if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[SX2H][#6;!$(C=O);!$(C=S)]")):
    selected_operators.append(smethylation_op)

tree = ETree.create(source=start_mol, target=target_mol)
tree.morphing_operators = tuple(dict.fromkeys(selected_operators))

class FindClosest:
    def __init__(self):
        self.closest_mol = None
        self.closest_distance = None
    def __call__(self, morph):
        if not self.closest_mol or self.closest_distance > morph.dist_to_target:
            self.closest_mol = morph
            self.closest_distance = morph.dist_to_target

closest_info = FindClosest()

while not tree.path_found:
    tree.generateMorphs()
    tree.sortMorphs()
    tree.filterMorphs()
    tree.extend()
    tree.prune()
    tree.traverse(closest_info)
        
    print(f"Generation #{tree.generation_count}")
    print(f"Molecules in tree: {tree.mol_count}")
    print(f"Closest to target: {closest_info.closest_mol.getSMILES()} (Distance: {closest_info.closest_distance:.4f})")
    print("-" * 40)
        
    if tree.path_found or tree.generation_count >= 5:
        break

print("\nSearch finished!")
if tree.path_found:
    print("SUCCESS.")


In [1]:
import random
from rdkit import Chem
from molpher.core import MolpherMol, MolpherAtom
from molpher.core.morphing.operators import MorphingOperator
from rdkit.Chem.EnumerateStereoisomers import EnumerateStereoisomers, StereoEnumerationOptions
from rdkit.Chem import rdChemReactions
from rdkit.Chem import rdmolops
from rdkit.Chem import Descriptors  
from molpher.core import ExplorationTree as ETree


class OxidizeAlcohol(MorphingOperator):
    def __init__(self):
        super(OxidizeAlcohol, self).__init__()
        self._name = "Oxidize Alcohol"
        self._target_atoms = [] 
        self.PATTERN = Chem.MolFromSmarts("[OX2H][#6X4;H1,H2]")

    def setOriginal(self, mol):
        super(OxidizeAlcohol, self).setOriginal(mol)
        self._target_atoms = []
        
        if not self.original:
            return
            
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: 
            return
            
        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
        
            self._target_atoms.append((match[0], match[1]))

    def morph(self):
        if not self.original: 
            return None
            
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: 
            return None
            
        if not self._target_atoms:
            return MolpherMol(other=rdkit_mol)
            
        idx_o, idx_c = random.choice(self._target_atoms)
        
        try:
            rw_mol = Chem.RWMol(rdkit_mol)
            
            if rw_mol.GetBondBetweenAtoms(idx_o, idx_c):
                rw_mol.RemoveBond(idx_o, idx_c)
            rw_mol.AddBond(idx_o, idx_c, Chem.BondType.DOUBLE)
            
            new_mol = rw_mol.GetMol()
    
            for idx in [idx_o, idx_c]:
                atom = new_mol.GetAtomWithIdx(idx)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
            
        except Exception as e:
            return MolpherMol(other=rdkit_mol)

    def getName(self):
        return self._name


class OxidizeAldehydeToAcid(MorphingOperator):
    def __init__(self):
        super(OxidizeAldehydeToAcid, self).__init__()
        self._name = "Oxidize Aldehyde to Acid"
        self._target_carbons = [] 
        self.PATTERN = Chem.MolFromSmarts("[CX3H1](=O)[#6,#1]")

    def setOriginal(self, mol):
        super(OxidizeAldehydeToAcid, self).setOriginal(mol)
        self._target_carbons = []
        
        if not self.original:
            return
            
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: 
            return
            
        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            self._target_carbons.append(match[0])

    def morph(self):
        if not self.original: 
            return None
            
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: 
            return None
        
        if not self._target_carbons:
            return MolpherMol(other=rdkit_mol)
            
        idx_c = random.choice(self._target_carbons)
        
        try:
            rw_mol = Chem.RWMol(rdkit_mol)
            
            new_o_idx = rw_mol.AddAtom(Chem.Atom(8))
            
            rw_mol.AddBond(idx_c, new_o_idx, Chem.BondType.SINGLE)
            
            new_mol = rw_mol.GetMol()
            
            for idx in [idx_c, new_o_idx]:
                atom = new_mol.GetAtomWithIdx(idx)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
        
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
            
        except Exception as e:
            return MolpherMol(other=rdkit_mol)
        
    def getName(self):
        return self._name


class AlkeneToAlcohol(MorphingOperator):
    def __init__(self):
        super(AlkeneToAlcohol, self).__init__()
        self._name = "Markovnikov Hydration"
        self._target_bonds = [] 
        self.PATTERN = Chem.MolFromSmarts("[CX3;H1,H2]=[CX3;H0,H1,H2]")

    def setOriginal(self, mol):
        super(AlkeneToAlcohol, self).setOriginal(mol)
        self._target_bonds = []
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return
        
        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            bond = rdkit_mol.GetBondBetweenAtoms(match[0], match[1])
            if bond and not bond.GetIsAromatic():
                self._target_bonds.append((match[0], match[1]))

    def morph(self):
        if not self.original: return None
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return None
        if not self._target_bonds: return MolpherMol(other=rdkit_mol)
            
        idx1, idx2 = random.choice(self._target_bonds)
        
        try:
            rw_mol = Chem.RWMol(rdkit_mol)
            
            bond = rw_mol.GetBondBetweenAtoms(idx1, idx2)
            if bond:
                bond.SetBondType(Chem.BondType.SINGLE)
            
            h1 = rw_mol.GetAtomWithIdx(idx1).GetTotalNumHs()
            h2 = rw_mol.GetAtomWithIdx(idx2).GetTotalNumHs()
            idx_with_oh = idx1 if h1 <= h2 else idx2
            
            oh_idx = rw_mol.AddAtom(Chem.Atom(8))
            rw_mol.AddBond(idx_with_oh, oh_idx, Chem.BondType.SINGLE)
            
            new_mol = rw_mol.GetMol()
            for idx in [idx1, idx2, oh_idx]:
                atom = new_mol.GetAtomWithIdx(idx)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            return MolpherMol(other=new_mol)
        except:
            return MolpherMol(other=rdkit_mol)
    
    def getName(self): return self._name


class AlcoholToAlkene(MorphingOperator):
    def __init__(self):
        super(AlcoholToAlkene, self).__init__()
        self._name = "Saytzeff Dehydration"
        self._target_groups = [] 
        self.PATTERN = Chem.MolFromSmarts("[OX2H][#6X4;H1,H2;!$(C(O)=O)]")

    def setOriginal(self, mol):
        super(AlcoholToAlkene, self).setOriginal(mol)
        self._target_groups = []
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return
        
        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            self._target_groups.append((match[0], match[1]))

    def morph(self):
        if not self.original: return None
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return None
        if not self._target_groups: return MolpherMol(other=rdkit_mol)
            
        oh_idx, alpha_idx = random.choice(self._target_groups)
        alpha_atom = rdkit_mol.GetAtomWithIdx(alpha_idx)
        
        beta_carbons = [a for a in alpha_atom.GetNeighbors() if a.GetAtomicNum() == 6 and a.GetHybridization() == Chem.HybridizationType.SP3]
        if not beta_carbons: return MolpherMol(other=rdkit_mol)

        beta_carbons.sort(key=lambda x: x.GetTotalNumHs())
        beta_idx = beta_carbons[0].GetIdx()
        
        try:
            rw_mol = Chem.RWMol(rdkit_mol)
            
            bond_ab = rw_mol.GetBondBetweenAtoms(alpha_idx, beta_idx)
            if bond_ab:
                bond_ab.SetBondType(Chem.BondType.DOUBLE)
                
            bond_oh = rw_mol.GetBondBetweenAtoms(oh_idx, alpha_idx)
            if bond_oh:
                rw_mol.RemoveBond(oh_idx, alpha_idx)
            
            new_mol = rw_mol.GetMol()
            
            for idx in [alpha_idx, beta_idx]:
                atom = new_mol.GetAtomWithIdx(idx)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
            
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            
            frags = Chem.GetMolFrags(new_mol, asMols=True)
            if frags:

                frags = sorted(frags, key=lambda x: x.GetNumAtoms(), reverse=True)
                final_mol = frags[0]
                
                clean_smiles = Chem.MolToSmiles(final_mol)
                return MolpherMol(clean_smiles)
                
            return MolpherMol(other=rdkit_mol)
        except:
            return MolpherMol(other=rdkit_mol)
    
    def getName(self): return self._name


class HeteroatomOxidation(MorphingOperator):
    def __init__(self):
        super(HeteroatomOxidation, self).__init__()
        self._name = "Heteroatom Oxidation (Phase I)"
        self._matches = []
        self.N_PATTERN = Chem.MolFromSmarts("[N;X3;H0;!$(N-C=O);!a](-[#6])-[#6]")
        self.S_THIOETHER = Chem.MolFromSmarts("[S;X2;H0;!a]")
        self.S_SULFOXIDE = Chem.MolFromSmarts("[S;X3;D3;H0](=O)")

    def setOriginal(self, mol):
        super(HeteroatomOxidation, self).setOriginal(mol)
        self._matches = []
        
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return
        
        if self.N_PATTERN is not None:
            for match in rdkit_mol.GetSubstructMatches(self.N_PATTERN):
                self._matches.append((match[0], "N"))

        if self.S_THIOETHER is not None:
            for match in rdkit_mol.GetSubstructMatches(self.S_THIOETHER):
                self._matches.append((match[0], "S_thio"))

        if self.S_SULFOXIDE is not None:
            for match in rdkit_mol.GetSubstructMatches(self.S_SULFOXIDE):
                self._matches.append((match[0], "S_sulf"))

    def morph(self):
        if not self.original: return None
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return None

        if not self._matches:
            return MolpherMol(other=rdkit_mol)

        target_idx, atom_type = random.choice(self._matches)

        try:
            rw_mol = Chem.RWMol(rdkit_mol)
            
            oxygen_idx = rw_mol.AddAtom(Chem.Atom(8))
            
            target_atom = rw_mol.GetAtomWithIdx(target_idx)
            ox_atom = rw_mol.GetAtomWithIdx(oxygen_idx)

            if atom_type in ["S_thio", "S_sulf"]:

                rw_mol.AddBond(target_idx, oxygen_idx, Chem.BondType.DOUBLE)
                
                target_atom.SetNoImplicit(False)
                target_atom.SetNumExplicitHs(0)
                
            elif atom_type == "N":
                
                rw_mol.AddBond(target_idx, oxygen_idx, Chem.BondType.SINGLE)
                target_atom.SetFormalCharge(1)
                ox_atom.SetFormalCharge(-1)
                ox_atom.SetNoImplicit(True)
                ox_atom.SetNumExplicitHs(0)
                target_atom.SetNoImplicit(False)
                target_atom.SetNumExplicitHs(0)

            new_mol = rw_mol.GetMol()
            
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self):
        return self._name


class HydrolyzeEster(MorphingOperator):
    def __init__(self):
        super(HydrolyzeEster, self).__init__()
        self._name = "Ester Hydrolysis (Generalized)"
        self._matches = []
        self.PATTERN = Chem.MolFromSmarts("[CX3](=O)[OX2][#6]")

    def setOriginal(self, mol):
        super(HydrolyzeEster, self).setOriginal(mol)
        self._matches = []

        if not self.original:
            return

        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None:
            return

        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)

        for match in matches:
            carbonyl_c_idx = match[0]   
            carbonyl_o_idx = match[1]  
            ester_o_idx = match[2]      
            alkoxy_c_idx = match[3]   

            carbonyl_c = rdkit_mol.GetAtomWithIdx(carbonyl_c_idx)
            alkoxy_atom = rdkit_mol.GetAtomWithIdx(alkoxy_c_idx)
            
            oxygen_neighbors = [
                nb for nb in carbonyl_c.GetNeighbors()
                if nb.GetAtomicNum() == 8
            ]
            if len(oxygen_neighbors) > 2:
                continue

            # EXCLUSION: tert-butyl esters
            
            carbon_neighbors = [
                nb for nb in alkoxy_atom.GetNeighbors()
                if nb.GetAtomicNum() == 6
            ]
            if len(carbon_neighbors) == 3:
                continue

            self._matches.append(
                (carbonyl_c_idx, carbonyl_o_idx, ester_o_idx, alkoxy_c_idx)
            )

    def morph(self):
        if not self.original:
            return None

        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None:
            return None

        if not self._matches:
            
            return MolpherMol(other=rdkit_mol)

        carbonyl_c_idx, carbonyl_o_idx, ester_o_idx, alkoxy_c_idx = random.choice(self._matches)

        try:
            rw_mol = Chem.RWMol(rdkit_mol)

            if rw_mol.GetBondBetweenAtoms(carbonyl_c_idx, ester_o_idx):
                rw_mol.RemoveBond(carbonyl_c_idx, ester_o_idx)
            else:
                return MolpherMol(other=rdkit_mol)

            new_oh_idx = rw_mol.AddAtom(Chem.Atom(8))
            rw_mol.AddBond(carbonyl_c_idx, new_oh_idx, Chem.BondType.SINGLE)
            
            new_mol = rw_mol.GetMol()

            fragments = Chem.GetMolFrags(new_mol, asMols=True, sanitizeFrags=False)
            if not fragments:
                return MolpherMol(other=rdkit_mol)

            processed_frags = []
            for frag in fragments:
                frag_rw = Chem.RWMol(frag)
                for atom in frag_rw.GetAtoms():
                    atom.SetNoImplicit(False)
                    atom.SetNumExplicitHs(0)

                frag_mol = frag_rw.GetMol()
                try:
                    frag_mol.UpdatePropertyCache(strict=False)
                    Chem.SanitizeMol(frag_mol)
                    processed_frags.append(frag_mol)
                except Exception:
                    continue

            if not processed_frags:
                return MolpherMol(other=rdkit_mol)

            benzene = Chem.MolFromSmarts("c1ccccc1")
            ring_fragments = [f for f in processed_frags if f.HasSubstructMatch(benzene)]

            if ring_fragments:
                largest_frag = max(ring_fragments, key=lambda m: Descriptors.MolWt(m))
            else:
                largest_frag = max(processed_frags, key=lambda m: Descriptors.MolWt(m))

            largest_frag.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(largest_frag, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(largest_frag, cleanIt=True, force=True)

            return MolpherMol(other=largest_frag)

        except Exception as e:
            print(f"[Debug Error]: {e}")
            return MolpherMol(other=rdkit_mol)

    def getName(self):
        return self._name


class NDealkylation(MorphingOperator):
    def __init__(self):
        super(NDealkylation, self).__init__()
        self._name = "N-Dealkylation (Phase I - Advanced)"
        self._target_bonds = []
        self.PATTERN = Chem.MolFromSmarts("[NX3;H0,H1;!a;!$(N-C=O)][CX4;H1,H2,H3]")

    def setOriginal(self, mol):
        super(NDealkylation, self).setOriginal(mol)
        self._target_bonds = []
        
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return
        
        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            self._target_bonds.append((match[0], match[1]))

    def morph(self):
        if not self.original: return None
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return None
        
        if not self._target_bonds:
            return MolpherMol(other=rdkit_mol)
            
        idx_n, idx_c = random.choice(self._target_bonds)
        
        try:
            rw_mol = Chem.RWMol(rdkit_mol)
            
            bond = rw_mol.GetBondBetweenAtoms(idx_n, idx_c)
            if bond:
                rw_mol.RemoveBond(idx_n, idx_c)
                
            new_mol = rw_mol.GetMol()
            
            atom_n = new_mol.GetAtomWithIdx(idx_n)
            atom_n.SetNoImplicit(False)
            atom_n.SetNumExplicitHs(0)
            
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            
            frags_mols = Chem.GetMolFrags(new_mol, asMols=True)
            
            if frags_mols:
                
                frags_indices = Chem.GetMolFrags(new_mol, asMols=False)
                frags_with_meta = list(zip(frags_mols, frags_indices))
                frags_with_meta = sorted(
                    frags_with_meta, 
                    key=lambda x: (idx_n in x[1], x[0].GetNumAtoms()), 
                    reverse=True
                )
                
                final_mol = frags_with_meta[0][0]
                
                Chem.AssignStereochemistry(final_mol, cleanIt=True, force=True)
                clean_smiles = Chem.MolToSmiles(final_mol)
                return MolpherMol(clean_smiles)
                
            return MolpherMol(other=rdkit_mol)
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name


class AromaticHydroxylation(MorphingOperator):
    def __init__(self):
        super(AromaticHydroxylation, self).__init__()
        self._name = "Aromatic Hydroxylation (Phase I - Regioselective)"
        self._matches = []
        self.AROMATIC_C = Chem.MolFromSmarts("[c;H1]")

    def _get_para_score(self, mol, c_idx):
        """ Υπολογισμός para-θέσης σε 6μελείς δακτυλίους """
        for ring in mol.GetRingInfo().AtomRings():
            if c_idx in ring and len(ring) == 6:
                for r_idx in ring:
                    r_atom = mol.GetAtomWithIdx(r_idx)
                    has_ex_neighbor = any(n.GetIdx() not in ring for n in r_atom.GetNeighbors())
                    
                    if has_ex_neighbor:
                        path = Chem.GetShortestPath(mol, r_idx, c_idx)
                        if len(path) == 4: # Απόσταση para
                            return 10
        return 1 

    def setOriginal(self, mol):
        super(AromaticHydroxylation, self).setOriginal(mol)
        self._matches = []
        
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return

        if self.AROMATIC_C is not None:
            matches = rdkit_mol.GetSubstructMatches(self.AROMATIC_C)
            scored_sites = []
            for match in matches:
                c_idx = match[0]
                score = self._get_para_score(rdkit_mol, c_idx)
                scored_sites.append((c_idx, score))
                
            if scored_sites:
                max_score = max(site[1] for site in scored_sites)
                self._matches = [site[0] for site in scored_sites if site[1] == max_score]

    def morph(self):
        if not self.original: return None
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return None

        if not self._matches:
            return MolpherMol(other=rdkit_mol)

        c_idx = random.choice(self._matches)
        
        try:
            rw_mol = Chem.RWMol(rdkit_mol)
            
            new_o_idx = rw_mol.AddAtom(Chem.Atom(8))
            c_atom = rw_mol.GetAtomWithIdx(c_idx)
            o_atom = rw_mol.GetAtomWithIdx(new_o_idx)
            
            c_atom.SetNoImplicit(False)
            c_atom.SetNumExplicitHs(0)
            o_atom.SetNoImplicit(False)
            o_atom.SetNumExplicitHs(0)
            
            rw_mol.AddBond(c_idx, new_o_idx, Chem.BondType.SINGLE)
            
            new_mol = rw_mol.GetMol()
            
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name

    
class AliphaticEpoxidation(MorphingOperator):
    def __init__(self):
        super(AliphaticEpoxidation, self).__init__()
        self._name = "Aliphatic Epoxidation (Phase I - Stereospecific)"
        self._target_bonds = []
        self.DOUBLE_BOND_PATTERN = Chem.MolFromSmarts("[CX3;!a]=[CX3;!a]")

    def setOriginal(self, mol):
        super(AliphaticEpoxidation, self).setOriginal(mol)
        self._target_bonds = []
        
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return

        if self.DOUBLE_BOND_PATTERN is not None:
            matches = rdkit_mol.GetSubstructMatches(self.DOUBLE_BOND_PATTERN)
            for match in matches:
                
                pair = tuple(sorted([match[0], match[1]]))
                if pair not in self._target_bonds:
                    self._target_bonds.append(pair)

    def morph(self):
        if not self.original: return None
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return None

        if not self._target_bonds:
            return MolpherMol(other=rdkit_mol)
        
        idx1, idx2 = random.choice(self._target_bonds)

        try:
            rw_mol = Chem.RWMol(rdkit_mol)
            
            bond = rw_mol.GetBondBetweenAtoms(idx1, idx2)
            if bond:
                bond.SetBondType(Chem.BondType.SINGLE)
            
            oxygen_idx = rw_mol.AddAtom(Chem.Atom(8))
            
            rw_mol.AddBond(idx1, oxygen_idx, Chem.BondType.SINGLE)
            rw_mol.AddBond(idx2, oxygen_idx, Chem.BondType.SINGLE)
            
            new_mol = rw_mol.GetMol()

            for idx in [idx1, idx2, oxygen_idx]:
                atom = new_mol.GetAtomWithIdx(idx)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
    
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self):
        return self._name


class AzoReduction(MorphingOperator):
    def __init__(self):
        super(AzoReduction, self).__init__()
        self._name = "Azo Reduction (Phase I - Safe Fragmentation)"
        self._matches = []
        self.AZO_PATTERN = Chem.MolFromSmarts("[C,c]-[N;!R]=[N;!R]-[C,c]")

    def setOriginal(self, mol):
        super(AzoReduction, self).setOriginal(mol)
        self._matches = []

        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return

        matches = rdkit_mol.GetSubstructMatches(self.AZO_PATTERN)
        for match in matches:
            self._matches.append((match[0], match[1], match[2], match[3]))

    def morph(self):
        if not self.original: return None
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return None

        if not self._matches:
            return MolpherMol(other=rdkit_mol)

        edit_mol = Chem.RWMol(rdkit_mol)
        c1_idx, n1_idx, n2_idx, c2_idx = random.choice(self._matches)

        try:
            edit_mol.RemoveBond(n1_idx, n2_idx)

            for idx in [n1_idx, n2_idx]:
                atom = edit_mol.GetAtomWithIdx(idx)
                atom.SetFormalCharge(0)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                for prop in list(atom.GetPropNames()):
                    atom.ClearProp(prop)

            new_mol = edit_mol.GetMol()
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            
            fragments = rdmolops.GetMolFrags(new_mol, asMols=True)
            if not fragments:
                return MolpherMol(other=rdkit_mol)

            chosen_frag = random.choice(fragments)
            
            Chem.AssignStereochemistry(chosen_frag, cleanIt=True, force=True)
            return MolpherMol(other=chosen_frag)
                
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name

    
class NitroReduction(MorphingOperator):
    def __init__(self):
        super(NitroReduction, self).__init__()
        self._name = "Nitro Reduction (Phase I - Safe)"
        self._target_nitrogens = []
        self.NITRO_PATTERN = Chem.MolFromSmarts("[C,c][N;X3](=[O,O-])~[O,O-]")

    def setOriginal(self, mol):
        super(NitroReduction, self).setOriginal(mol)
        self._target_nitrogens = []

        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return

        matches = rdkit_mol.GetSubstructMatches(self.NITRO_PATTERN)
        for match in matches:
            if match[1] not in self._target_nitrogens:
                self._target_nitrogens.append(match[1])

    def morph(self):
        if not self.original: return None
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return None

        if not self._target_nitrogens:
            return MolpherMol(other=rdkit_mol)

        n_idx = random.choice(self._target_nitrogens)

        try:
            rw_mol = Chem.RWMol(rdkit_mol)
            n_atom = rw_mol.GetAtomWithIdx(n_idx)
            
            o_indices = [neighbor.GetIdx() for neighbor in n_atom.GetNeighbors() if neighbor.GetAtomicNum() == 8]
            
            for o_idx in o_indices:
                bond = rw_mol.GetBondBetweenAtoms(n_idx, o_idx)
                if bond:
                    rw_mol.RemoveBond(n_idx, o_idx)
            
            n_atom.SetFormalCharge(0)
            n_atom.SetNoImplicit(False)
            n_atom.SetNumExplicitHs(2)
            for prop in list(n_atom.GetPropNames()):
                n_atom.ClearProp(prop)
            
            new_mol = rw_mol.GetMol()
            new_mol.UpdatePropertyCache(strict=False)
            
            atoms_to_remove = []
            for atom in new_mol.GetAtoms():
                if atom.GetAtomicNum() == 8 and atom.GetDegree() == 0:
                    atoms_to_remove.append(atom.GetIdx())

            edit = Chem.RWMol(new_mol)
            for idx in sorted(atoms_to_remove, reverse=True):
                edit.RemoveAtom(idx)
            new_mol = edit.GetMol()

            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            return MolpherMol(other=new_mol)

        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name


class AldehydeReduction(MorphingOperator):
    def __init__(self):
        super(AldehydeReduction, self).__init__()
        self._name = "Aldehyde Reduction (Phase I - Safe)"
        self._matches = []
        self.ALDEHYDE_PATTERN = Chem.MolFromSmarts("[CX3H1;!$(C(=O)[O,N,S,F,Cl,Br,I])]=[OX1]")

    def setOriginal(self, mol):
        super(AldehydeReduction, self).setOriginal(mol)
        self._matches = []

        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return

        matches = rdkit_mol.GetSubstructMatches(self.ALDEHYDE_PATTERN)
        for match in matches:
            self._matches.append((match[0], match[1]))

    def morph(self):
        if not self.original: return None
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return None

        if not self._matches:
            return MolpherMol(other=rdkit_mol)

        edit_mol = Chem.RWMol(rdkit_mol)
        c_idx, o_idx = random.choice(self._matches)

        try:
            bond = edit_mol.GetBondBetweenAtoms(c_idx, o_idx)
            if bond is None: return MolpherMol(other=rdkit_mol)

            bond.SetBondType(Chem.BondType.SINGLE)

            for idx in [c_idx, o_idx]:
                atom = edit_mol.GetAtomWithIdx(idx)
                atom.SetFormalCharge(0)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                for prop in list(atom.GetPropNames()):
                    atom.ClearProp(prop)
                atom.UpdatePropertyCache(strict=False)

            new_mol = edit_mol.GetMol()
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name


class KetoneReduction(MorphingOperator):
    def __init__(self):
        super(KetoneReduction, self).__init__()
        self._name = "Ketone Reduction (Phase I - Stereospecific Enumerate)"
        self._matches = []

    def setOriginal(self, mol):
        super(KetoneReduction, self).setOriginal(mol)
        self._matches = []

        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return
        
        ketone_pattern = Chem.MolFromSmarts("[CX3;$(C(=O)(-[#6])-[#6])]=O")
        matches = rdkit_mol.GetSubstructMatches(ketone_pattern)
        for match in matches:
            self._matches.append((match[0], match[1]))

    def morph(self):
        if not self.original: return None
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return None

        if not self._matches:
            return MolpherMol(other=rdkit_mol)

        edit_mol = Chem.RWMol(rdkit_mol)
        carbonyl_idx, oxygen_idx = random.choice(self._matches)

        try:
            bond = edit_mol.GetBondBetweenAtoms(carbonyl_idx, oxygen_idx)
            if bond is None: return MolpherMol(other=rdkit_mol)

            bond.SetBondType(Chem.BondType.SINGLE)

            for idx in [carbonyl_idx, oxygen_idx]:
                atom = edit_mol.GetAtomWithIdx(idx)
                atom.SetFormalCharge(0)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                atom.SetChiralTag(Chem.ChiralType.CHI_UNSPECIFIED)
                if atom.HasProp('_CIPCode'): atom.ClearProp('_CIPCode')
                for prop in list(atom.GetPropNames()): atom.ClearProp(prop)
                atom.UpdatePropertyCache(strict=False)

            new_mol = edit_mol.GetMol()
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)

            options = StereoEnumerationOptions(onlyUnassigned=True)
            isomers = list(EnumerateStereoisomers(new_mol, options=options))

            if isomers:
                chosen_iso = random.choice(isomers)
                Chem.SanitizeMol(chosen_iso)
                Chem.AssignStereochemistry(chosen_iso, cleanIt=True, force=True)
                return MolpherMol(other=chosen_iso)

            return MolpherMol(other=new_mol)
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name


class AlkenylReduction(MorphingOperator):
    def __init__(self):
        super(AlkenylReduction, self).__init__()
        self._name = "Alkenyl Reduction (Phase I - Ring Safe)"
        self._target_bonds = []
        self.PATTERN = Chem.MolFromSmarts("[CX3;!a]=[CX3;!a]")

    def setOriginal(self, mol):
        super(AlkenylReduction, self).setOriginal(mol)
        self._target_bonds = []
        
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return

        if self.PATTERN is not None:
            matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
            for match in matches:
                pair = tuple(sorted([match[0], match[1]]))
                if pair not in self._target_bonds:
                    self._target_bonds.append(pair)

    def morph(self):
        if not self.original: return None
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return None

        if not self._target_bonds:
            return MolpherMol(other=rdkit_mol)
        
        idx1, idx2 = random.choice(self._target_bonds)

        try:
            rw_mol = Chem.RWMol(rdkit_mol)
            
            bond = rw_mol.GetBondBetweenAtoms(idx1, idx2)
            if bond:
                bond.SetBondType(Chem.BondType.SINGLE)
                bond.SetStereo(Chem.BondStereo.STEREONONE)
            
            for idx in [idx1, idx2]:
                atom = rw_mol.GetAtomWithIdx(idx)
                atom.SetFormalCharge(0)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                atom.SetChiralTag(Chem.ChiralType.CHI_UNSPECIFIED)
                if atom.HasProp('_CIPCode'): 
                    atom.ClearProp('_CIPCode')
                for prop in list(atom.GetPropNames()):
                    atom.ClearProp(prop)
                atom.UpdatePropertyCache(strict=False)
            
            new_mol = rw_mol.GetMol()
            
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self):
        return self._name


class LactamFormation(MorphingOperator):
    def __init__(self):
        super(LactamFormation, self).__init__()
        self._name = "Lactam Formation (Phase I - Ring Oxidation Safe)"
        self._matches = []
        self.PATTERN = Chem.MolFromSmarts("[NX3;R;!$(N-C=O);!a]-[CX4;R;H2;!$(C-O)]")

    def setOriginal(self, mol):
        super(LactamFormation, self).setOriginal(mol)
        self._matches = []

        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return

        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            self._matches.append((match[0], match[1]))

    def morph(self):
        if not self.original: return None
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return None

        if not self._matches:
            return MolpherMol(other=rdkit_mol)

        n_idx, c_idx = random.choice(self._matches)

        try:
            rw_mol = Chem.RWMol(rdkit_mol)
            
            o_atom = Chem.Atom(8)
            o_atom.SetFormalCharge(0)
            o_idx = rw_mol.AddAtom(o_atom)
            
            rw_mol.AddBond(c_idx, o_idx, Chem.BondType.DOUBLE)
            
            for idx in [n_idx, c_idx, o_idx]:
                atom = rw_mol.GetAtomWithIdx(idx)
                atom.SetFormalCharge(0)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                for prop in list(atom.GetPropNames()):
                    atom.ClearProp(prop)
                atom.UpdatePropertyCache(strict=False)
            
            new_mol = rw_mol.GetMol()
            
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self):
        return self._name


class AcylGlucuronidation(MorphingOperator):
    def __init__(self):
        super(AcylGlucuronidation, self).__init__()
        self._name = "Acyl Glucuronidation (Carboxylic Acids)"
        self._target_oxygens = []
        self.PATTERN = Chem.MolFromSmarts("[CX3](=O)[OX2H]")
        self.GLUCURONIDE_TEMPLATE = Chem.MolFromSmiles("C1(O)O[C@H](C(=O)O)[C@@H](O)[C@H](O)[C@@H]1O")

    def setOriginal(self, mol):
        super(AcylGlucuronidation, self).setOriginal(mol)
        self._target_oxygens = []
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return
        
        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            self._target_oxygens.append(match[2])

    def morph(self):
        if not self.original or not self._target_oxygens:
            return MolpherMol(other=self.original.asRDMol())
        
        rdkit_mol = self.original.asRDMol()
        try:
            target_o_idx = random.choice(self._target_oxygens)
            
            combined = Chem.CombineMols(rdkit_mol, self.GLUCURONIDE_TEMPLATE)
            rw_combined = Chem.RWMol(combined)
            
            c_sugar_idx = rdkit_mol.GetNumAtoms()
            oh_sugar_idx = c_sugar_idx + 1
            
            rw_combined.AddBond(target_o_idx, c_sugar_idx, Chem.BondType.SINGLE)
        
            rw_combined.RemoveAtom(oh_sugar_idx)
        
            new_mol = rw_combined.GetMol()
            
            for idx in [target_o_idx, c_sugar_idx]:
                atom = new_mol.GetAtomWithIdx(idx)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
            
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name

   
class AlcoholPhenolGlucuronidation(MorphingOperator):
    def __init__(self):
        super(AlcoholPhenolGlucuronidation, self).__init__()
        self._name = "O-Glucuronidation (Alcohols/Phenols)"
        self._target_oxygens = []
        
        # SMARTS: Επιλέγει [OX2H] που συνδέεται με άνθρακα, ο οποίος ΔΕΝ είναι καρβονύλιο
        self.PATTERN = Chem.MolFromSmarts("[#6;!$(C=O);!$(C=C)][OX2H]")
        # Template: β-D-glucuronide με SMILES όπου ο C1 (ανωμερής) είναι το 1ο άτομο (index 0)
        # SMILES: C1([OH])O[C@H](C(=O)O)[C@@H](O)[C@H](O)[C@@H]1O
        # Με αυτό το SMILES: 
        # index 0 -> ο C1 που θα ενωθεί με το υπόλοιπο μόριο
        # index 1 -> το -OH του C1 το οποίο ΠΡΕΠΕΙ να αφαιρεθεί
        self.GLUCURONIDE_TEMPLATE = Chem.MolFromSmiles("C1(O)O[C@H](C(=O)O)[C@@H](O)[C@H](O)[C@@H]1O")

    def setOriginal(self, mol):
        super(AlcoholPhenolGlucuronidation, self).setOriginal(mol)
        self._target_oxygens = []
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return

        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            # Το [OX2H] είναι το δεύτερο άτομο στο pattern (index 1)
            self._target_oxygens.append(match[1])

    def morph(self):
        if not self.original or not self._target_oxygens:
            return MolpherMol(other=self.original.asRDMol())

        rdkit_mol = self.original.asRDMol()
        try:
            target_o_idx = random.choice(self._target_oxygens)
            
            combined = Chem.CombineMols(rdkit_mol, self.GLUCURONIDE_TEMPLATE)
            rw_combined = Chem.RWMol(combined)
            c_sugar_idx = rdkit_mol.GetNumAtoms() 
            oh_sugar_idx = c_sugar_idx + 1 
            
            rw_combined.AddBond(target_o_idx, c_sugar_idx, Chem.BondType.SINGLE)
            rw_combined.RemoveAtom(oh_sugar_idx)
            
            new_mol = rw_combined.GetMol()
            
            for idx in [target_o_idx, c_sugar_idx]:
                atom = new_mol.GetAtomWithIdx(idx)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
            
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name


    
class AlcoholPhenolSulfation(MorphingOperator):
    def __init__(self):
        super(AlcoholPhenolSulfation, self).__init__()
        self._name = "O-Sulfation (Alcohols/Phenols)"
        self._target_atoms = []
        self.PATTERN = Chem.MolFromSmarts("[OX2H][#6;!$(C=O)]")

    def setOriginal(self, mol):
        super(AlcoholPhenolSulfation, self).setOriginal(mol)
        self._target_atoms = []

        if not self.original:
            return

        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None:
            return

        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            self._target_atoms.append(match[0])

    def morph(self):
        if not self.original:
            return None

        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None:
            return None

        if not self._target_atoms:
            return MolpherMol(other=rdkit_mol)

        oxygen_idx = random.choice(self._target_atoms)

        try:
            rw_mol = Chem.RWMol(rdkit_mol)

            # Εισαγωγή ουδέτερης ομάδας -SO3H
            sulfur_idx = rw_mol.AddAtom(Chem.Atom(16)) # S
            rw_mol.AddBond(oxygen_idx, sulfur_idx, Chem.BondType.SINGLE)

            o1_idx = rw_mol.AddAtom(Chem.Atom(8)) # =O
            rw_mol.AddBond(sulfur_idx, o1_idx, Chem.BondType.DOUBLE)

            o2_idx = rw_mol.AddAtom(Chem.Atom(8)) # =O
            rw_mol.AddBond(sulfur_idx, o2_idx, Chem.BondType.DOUBLE)

            o3_idx = rw_mol.AddAtom(Chem.Atom(8)) # -OH (Ουδέτερο)
            rw_mol.AddBond(sulfur_idx, o3_idx, Chem.BondType.SINGLE)

            new_mol = rw_mol.GetMol()

            for idx in [oxygen_idx, sulfur_idx, o1_idx, o2_idx, o3_idx]:
                atom = new_mol.GetAtomWithIdx(idx)
                if atom.GetAtomicNum() != 16: # Αφήνουμε το Θείο να διαχειριστεί το σθένος του (6)
                    atom.SetNoImplicit(False)
                    atom.SetNumExplicitHs(0)

            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)

            return MolpherMol(other=new_mol)

        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self):
        return self._name


class NitrogenSulfation(MorphingOperator):
    def __init__(self):
        super(NitrogenSulfation, self).__init__()
        self._name = "N-Sulfation (Amines 1°, 2°, 3° & Aromatic)"
        self._target_atoms = []
        self.PATTERN = Chem.MolFromSmarts("[N;X3;!$(N-C=O);!$(N-C(=O)N);!$(N-C(=O)O)]")

    def setOriginal(self, mol):
        super(NitrogenSulfation, self).setOriginal(mol)
        self._target_atoms = []
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return
        
        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            self._target_atoms.append(match[0])

    def morph(self):
        if not self.original: return None
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return None
        
        if not self._target_atoms:
            return MolpherMol(other=rdkit_mol)
        
        nitrogen_idx = random.choice(self._target_atoms)
        
        try:
            rw_mol = Chem.RWMol(rdkit_mol)
            is_tertiary = (rw_mol.GetAtomWithIdx(nitrogen_idx).GetTotalNumHs() == 0)
            
            sulfur_idx = rw_mol.AddAtom(Chem.Atom(16)) # S
            rw_mol.AddBond(nitrogen_idx, sulfur_idx, Chem.BondType.SINGLE)
            
            o1_idx = rw_mol.AddAtom(Chem.Atom(8)) # =O
            rw_mol.AddBond(sulfur_idx, o1_idx, Chem.BondType.DOUBLE)
            
            o2_idx = rw_mol.AddAtom(Chem.Atom(8)) # =O
            rw_mol.AddBond(sulfur_idx, o2_idx, Chem.BondType.DOUBLE)
            
            o3_idx = rw_mol.AddAtom(Chem.Atom(8)) # -OH
            rw_mol.AddBond(sulfur_idx, o3_idx, Chem.BondType.SINGLE)
            
            new_mol = rw_mol.GetMol()
            
            for idx in [o1_idx, o2_idx, o3_idx]:
                atom = new_mol.GetAtomWithIdx(idx)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                atom.SetFormalCharge(0)
            
            n_atom = new_mol.GetAtomWithIdx(nitrogen_idx)
            n_atom.SetNoImplicit(False)
            n_atom.SetNumExplicitHs(0)
            
            if is_tertiary:
                
                n_atom.SetFormalCharge(1)
            else:
                
                n_atom.SetFormalCharge(0)
                
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
            
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name
    

class AminoAcidConjugation(MorphingOperator):
    def __init__(self):
        super(AminoAcidConjugation, self).__init__()
        self._name = "Amino Acid Conjugation (Gly/Tau/Gln)"
        self._target_atoms = []
        self.PATTERN = Chem.MolFromSmarts("[CX3](=O)[OX2H]") 
        # Σε όλα, το Άζωτο (Ν) που θα επιτεθεί είναι το ΠΡΩΤΟ άτομο (index 0)
        self.TEMPLATES = {
            "Glycine": Chem.MolFromSmiles("NCC(=O)O"),
            "Taurine": Chem.MolFromSmiles("NCCS(=O)(=O)O"),
            "Glutamine": Chem.MolFromSmiles("N[C@@H](CCC(=O)N)C(=O)O") # Διατήρηση στερεοχημείας
        }

    def setOriginal(self, mol):
        super(AminoAcidConjugation, self).setOriginal(mol)
        self._target_atoms = []
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return
        
        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            # Κρατάμε το index του Καρβονυλικού Άνθρακα (index 0) και του -OH (index 2)
            self._target_atoms.append((match[0], match[2]))

    def morph(self):
        if not self.original or not self._target_atoms:
            return MolpherMol(other=self.original.asRDMol())
        
        rdkit_mol = self.original.asRDMol()
        try:
            chosen_c_idx, chosen_oh_idx = random.choice(self._target_atoms)
            template_name = random.choice(list(self.TEMPLATES.keys()))
            template_mol = self.TEMPLATES[template_name]
            combined = Chem.CombineMols(rdkit_mol, template_mol)
            rw_combined = Chem.RWMol(combined)
            n_amino_idx = rdkit_mol.GetNumAtoms()
            rw_combined.AddBond(chosen_c_idx, n_amino_idx, Chem.BondType.SINGLE)
            rw_combined.RemoveAtom(chosen_oh_idx)
            
            new_mol = rw_combined.GetMol()
            
            # Λόγω του RemoveAtom, ο index του n_amino_idx μετατοπίστηκε κατά -1 
            # αν ο chosen_oh_idx ήταν μικρότερος, αλλά για ασφάλεια κάνουμε reset στο στοχευμένο καρβονύλιο 
            # και στο άζωτο σαρώνοντας το γράφημα.
            for atom in new_mol.GetAtoms():
                if atom.GetAtomicNum() in [6, 7]: # Άνθρακας καρβονυλίου και Άζωτο αμιδίου
                    atom.SetNoImplicit(False)
                    atom.SetNumExplicitHs(0)
                    atom.SetFormalCharge(0)
            
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
            
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name

    
class NAcetylation(MorphingOperator):
    def __init__(self):
        super(NAcetylation, self).__init__()
        self._name = "N-acetylation (Primary Amines)"
        self._target_atoms = []
        self.PATTERN = Chem.MolFromSmarts("[N;H2;!$(N-C=O);!$(N-S(=O)=O);!$(NN)]")

    def setOriginal(self, mol):
        super(NAcetylation, self).setOriginal(mol)
        self._target_atoms = []
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return
        
        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            self._target_atoms.append(match[0])

    def morph(self):
        if not self.original or not self._target_atoms:
            return MolpherMol(other=self.original.asRDMol())
        
        rdkit_mol = self.original.asRDMol()
        try:
            chosen_idx = random.choice(self._target_atoms)
            rw_mol = Chem.RWMol(rdkit_mol)
            
            c_carbonyl_idx = rw_mol.AddAtom(Chem.Atom(6))
            rw_mol.AddBond(chosen_idx, c_carbonyl_idx, Chem.BondType.SINGLE)
            
            o_carbonyl_idx = rw_mol.AddAtom(Chem.Atom(8))
            rw_mol.AddBond(c_carbonyl_idx, o_carbonyl_idx, Chem.BondType.DOUBLE)
            
            c_methyl_idx = rw_mol.AddAtom(Chem.Atom(6))
            rw_mol.AddBond(c_carbonyl_idx, c_methyl_idx, Chem.BondType.SINGLE)
            
            new_mol = rw_mol.GetMol()
            
            for idx in [chosen_idx, c_carbonyl_idx, o_carbonyl_idx, c_methyl_idx]:
                atom = new_mol.GetAtomWithIdx(idx)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                atom.SetFormalCharge(0)
                
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
            
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name


class HydrazineAcetylation(MorphingOperator):
    def __init__(self):
        super(HydrazineAcetylation, self).__init__()
        self._name = "Hydrazine acetylation"
        self._target_atoms = []
        self.PATTERN = Chem.MolFromSmarts("[NX3;!$(N-C=O)][NX3;H2;!$(N-C=O)]")

    def setOriginal(self, mol):
        super(HydrazineAcetylation, self).setOriginal(mol)
        self._target_atoms = []
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return
        
        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            # Κρατάμε ΜΟΝΟ το index του ακραίου αζώτου [-NH2] 
            self._target_atoms.append(match[1])

    def morph(self):
        if not self.original or not self._target_atoms:
            return MolpherMol(other=self.original.asRDMol())
        
        rdkit_mol = self.original.asRDMol()
        try:
            terminal_n_idx = random.choice(self._target_atoms)
            rw_mol = Chem.RWMol(rdkit_mol)
            
            c_carbonyl_idx = rw_mol.AddAtom(Chem.Atom(6))
            rw_mol.AddBond(terminal_n_idx, c_carbonyl_idx, Chem.BondType.SINGLE)
            
            o_carbonyl_idx = rw_mol.AddAtom(Chem.Atom(8))
            rw_mol.AddBond(c_carbonyl_idx, o_carbonyl_idx, Chem.BondType.DOUBLE)
            
            c_methyl_idx = rw_mol.AddAtom(Chem.Atom(6))
            rw_mol.AddBond(c_carbonyl_idx, c_methyl_idx, Chem.BondType.SINGLE)
            
            new_mol = rw_mol.GetMol()
            
            for idx in [terminal_n_idx, c_carbonyl_idx, o_carbonyl_idx, c_methyl_idx]:
                atom = new_mol.GetAtomWithIdx(idx)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                atom.SetFormalCharge(0)
                
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
            
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name

  
class HydrazideAcetylation(MorphingOperator):
    def __init__(self):
        super(HydrazideAcetylation, self).__init__()
        self._name = "Hydrazide acetylation"
        self._target_atoms = []
        self.PATTERN = Chem.MolFromSmarts("[CX3](=O)[NX3;H1][NX3;H2;!$(N-C=O)]")

    def setOriginal(self, mol):
        super(HydrazideAcetylation, self).setOriginal(mol)
        self._target_atoms = []
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return
        
        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            # Κρατάμε το index του ακραίου αζώτου [-NH2] (index 3)
            self._target_atoms.append(match[3])

    def morph(self):
        if not self.original or not self._target_atoms:
            return MolpherMol(other=self.original.asRDMol())
        
        rdkit_mol = self.original.asRDMol()
        try:
            terminal_n_idx = random.choice(self._target_atoms)
            rw_mol = Chem.RWMol(rdkit_mol)
            
            c_carbonyl_idx = rw_mol.AddAtom(Chem.Atom(6))
            rw_mol.AddBond(terminal_n_idx, c_carbonyl_idx, Chem.BondType.SINGLE)
            
            o_carbonyl_idx = rw_mol.AddAtom(Chem.Atom(8))
            rw_mol.AddBond(c_carbonyl_idx, o_carbonyl_idx, Chem.BondType.DOUBLE)
            
            c_methyl_idx = rw_mol.AddAtom(Chem.Atom(6))
            rw_mol.AddBond(c_carbonyl_idx, c_methyl_idx, Chem.BondType.SINGLE)
            
            new_mol = rw_mol.GetMol()
 
            for idx in [terminal_n_idx, c_carbonyl_idx, o_carbonyl_idx, c_methyl_idx]:
                atom = new_mol.GetAtomWithIdx(idx)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                atom.SetFormalCharge(0)
                
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
            
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name
    

class OMethylation(MorphingOperator):
    def __init__(self):
        super(OMethylation, self).__init__()
        self._name = "O-methylation (Alcohols/Phenols)"
        self._target_atoms = []
        self.PATTERN = Chem.MolFromSmarts("[OX2H][#6;!$(C=O);!$(C=C);!$(C=N)]")

    def setOriginal(self, mol):
        super(OMethylation, self).setOriginal(mol)
        self._target_atoms = []
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return
        
        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            self._target_atoms.append(match[0])

    def morph(self):
        if not self.original or not self._target_atoms:
            return MolpherMol(other=self.original.asRDMol())
        
        rdkit_mol = self.original.asRDMol()
        try:
            target_o_idx = random.choice(self._target_atoms)
            rw_mol = Chem.RWMol(rdkit_mol)
            
            c_methyl_idx = rw_mol.AddAtom(Chem.Atom(6))
            rw_mol.AddBond(target_o_idx, c_methyl_idx, Chem.BondType.SINGLE)
            
            new_mol = rw_mol.GetMol()
            
            for idx in [target_o_idx, c_methyl_idx]:
                atom = new_mol.GetAtomWithIdx(idx)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                atom.SetFormalCharge(0)
                
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
            
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name

  
class NMethylation(MorphingOperator):
    def __init__(self):
        super(NMethylation, self).__init__()
        self._name = "N-methylation (Primary Amines)"
        self._target_atoms = []
        
        # Ενισχυμένο SMARTS Pattern για Πρωτοταγείς Αμίνες:
        # Ζητάμε άζωτο με 2 υδρογόνα [N;H2]
        # Αποκλείουμε: Αμίδια (!$(N-C=O)), Υδραζίνες (!$(NN)), 
        # Σουλφοναμίδια (!$(N-S(=O)=O)), Ουρίες/Καρβαμιδικά (!$(N-C(=O)))
        self.PATTERN = Chem.MolFromSmarts("[N;H2;!$(N-C=O);!$(N-S(=O)=O);!$(NN)]")

    def setOriginal(self, mol):
        super(NMethylation, self).setOriginal(mol)
        self._target_atoms = []
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return
        
        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            self._target_atoms.append(match[0])

    def morph(self):
        if not self.original or not self._target_atoms:
            return MolpherMol(other=self.original.asRDMol())
        
        rdkit_mol = self.original.asRDMol()
        try:
            target_n_idx = random.choice(self._target_atoms)
            rw_mol = Chem.RWMol(rdkit_mol)
            
            # Προσθήκη του Άνθρακα του Μεθυλίου (-CH3)
            c_methyl_idx = rw_mol.AddAtom(Chem.Atom(6))
            rw_mol.AddBond(target_n_idx, c_methyl_idx, Chem.BondType.SINGLE)
            
            new_mol = rw_mol.GetMol()
            
            # Reset ιδιοτήτων στα άτομα που τροποποιήθηκαν
            for idx in [target_n_idx, c_methyl_idx]:
                atom = new_mol.GetAtomWithIdx(idx)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                atom.SetFormalCharge(0)
                
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
            
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name
    

class SMethylation(MorphingOperator):
    def __init__(self):
        super(SMethylation, self).__init__()
        self._name = "S-methylation (Thiols)"
        self._target_atoms = []
        self.PATTERN = Chem.MolFromSmarts("[SX2H][#6;!$(C=O);!$(C=S)]")

    def setOriginal(self, mol):
        super(SMethylation, self).setOriginal(mol)
        self._target_atoms = []
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return
        
        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            # Κρατάμε το index του θείου [-SH] (το πρώτο άτομο του match, index 0)
            self._target_atoms.append(match[0])

    def morph(self):
        if not self.original or not self._target_atoms:
            return MolpherMol(other=self.original.asRDMol())
        
        rdkit_mol = self.original.asRDMol()
        try:
            target_s_idx = random.choice(self._target_atoms)
            rw_mol = Chem.RWMol(rdkit_mol)
            
            c_methyl_idx = rw_mol.AddAtom(Chem.Atom(6))
            rw_mol.AddBond(target_s_idx, c_methyl_idx, Chem.BondType.SINGLE)
            
            new_mol = rw_mol.GetMol()
            
            for idx in [target_s_idx, c_methyl_idx]:
                atom = new_mol.GetAtomWithIdx(idx)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                atom.SetFormalCharge(0)
                
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
            
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name

    
oxidize_alcohol_op = OxidizeAlcohol()
oxidize_aldehyde_op = OxidizeAldehydeToAcid() 
op_hydration = AlkeneToAlcohol()
op_dehydration = AlcoholToAlkene()
hetero_op = HeteroatomOxidation()
hydrolize_ester = HydrolyzeEster()
dealk_op = NDealkylation()
hydroxylation_op = AromaticHydroxylation()
epox_op = AliphaticEpoxidation()
azo_op = AzoReduction()
nitro_op = NitroReduction()
aldehyde_op = AldehydeReduction()
ketone_op = KetoneReduction()
alkene_op = AlkenylReduction()
lactam_op = LactamFormation()
acyl_glucuronidation = AcylGlucuronidation()
alph_glucoronidation = AlcoholPhenolGlucuronidation()
sulfation_op = AlcoholPhenolSulfation()  
nitrogen_sulfation = NitrogenSulfation()
amino_op = AminoAcidConjugation()
nacet_op = NAcetylation()
hydrazine_acetylation_op = HydrazineAcetylation()  
hydrazide_acetylation_op = HydrazideAcetylation()
omethylation_op = OMethylation()  
nmethylation_op = NMethylation()
smethylation_op = SMethylation()  


start_smiles = input("Δώσε SMILES για το start molecule: ")
target_smiles = input("Δώσε SMILES για το target molecule: ")
start_mol = MolpherMol(start_smiles)
target_mol = MolpherMol(target_smiles)
rdkit_start = start_mol.asRDMol()

selected_operators = []

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[OX2H][#6X4;H1,H2]")):
    selected_operators.append(oxidize_alcohol_op)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[CX3H1](=O)[#6,#1]")):
    selected_operators.append(oxidize_aldehyde_op)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[CX3;H1,H2]=[CX3;H0,H1,H2]")):
    selected_operators.append(op_hydration)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[OX2H][#6X4;H1,H2;!$(C(O)=O)]")):
    selected_operators.append(op_dehydration)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[N;X3;H0;!$(N-C=O);!a](-[#6])-[#6]")):
    selected_operators.append(hetero_op)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[S;X2;H0;!a]")):
    selected_operators.append(hetero_op)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[S;X3;D3;H0](=O)")):
    selected_operators.append(hetero_op)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[CX3](=O)[OX2][#6]")):
    selected_operators.append(hydrolize_ester)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[NX3;H0,H1;!a;!$(N-C=O)][CX4;H1,H2,H3]")):
    selected_operators.append(dealk_op)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[c;H1]")):
    selected_operators.append(hydroxylation_op)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[CX3;!a]=[CX3;!a]")):
    selected_operators.append(epox_op)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[C,c]-[N;!R]=[N;!R]-[C,c]")):
    selected_operators.append(azo_op)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[C,c][N;X3](=[O,O-])~[O,O-]")):
    selected_operators.append(nitro_op)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[CX3H1;!$(C(=O)[O,N,S,F,Cl,Br,I])]=[OX1]")):
    selected_operators.append(aldehyde_op)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[CX3;$(C(=O)(-[#6])-[#6])]=O")):
    selected_operators.append(ketone_op)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[CX3;!a]=[CX3;!a]")):
    selected_operators.append(alkene_op)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[NX3;R;!$(N-C=O);!a]-[CX4;R;H2;!$(C-O)]")):
    selected_operators.append(lactam_op)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[CX3](=O)[OX2H]")):
    selected_operators.append(acyl_glucuronidation)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[#6;!$(C=O);!$(C=C)][OX2H]")):
    selected_operators.append(alph_glucoronidation)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[OX2H][#6;!$(C=O)]")):
    selected_operators.append(sulfation_op)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[N;X3;!$(N-C=O);!$(N-C(=O)N);!$(N-C(=O)O)]")):
    selected_operators.append(nitrogen_sulfation)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[CX3](=O)[OX2H]")):
    selected_operators.append(amino_op)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[N;H2;!$(N-C=O);!$(N-S(=O)=O);!$(NN)]")):
    selected_operators.append(nacet_op)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[NX3;!$(N-C=O)][NX3;H2;!$(N-C=O)]")):
    selected_operators.append(hydrazine_acetylation_op)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[CX3](=O)[NX3;H1][NX3;H2;!$(N-C=O)]")):
    selected_operators.append(hydrazide_acetylation_op)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[OX2H][#6;!$(C=O);!$(C=C);!$(C=N)]")):
    selected_operators.append(omethylation_op)
    
if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[N;H2;!$(N-C=O);!$(N-S(=O)=O);!$(NN)]")):
    selected_operators.append(nmethylation_op)
    
if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[SX2H][#6;!$(C=O);!$(C=S)]")):
    selected_operators.append(smethylation_op)

tree = ETree.create(source=start_mol, target=target_mol)
tree.morphing_operators = tuple(dict.fromkeys(selected_operators))

class FindClosest:
    def __init__(self):
        self.closest_mol = None
        self.closest_distance = None
    def __call__(self, morph):
        if not self.closest_mol or self.closest_distance > morph.dist_to_target:
            self.closest_mol = morph
            self.closest_distance = morph.dist_to_target

closest_info = FindClosest()

while not tree.path_found:
    tree.generateMorphs()
    tree.sortMorphs()
    tree.filterMorphs()
    tree.extend()
    tree.prune()
    tree.traverse(closest_info)
        
    print(f"Generation #{tree.generation_count}")
    print(f"Molecules in tree: {tree.mol_count}")
    print(f"Closest to target: {closest_info.closest_mol.getSMILES()} (Distance: {closest_info.closest_distance:.4f})")
    print("-" * 40)
        
    if tree.path_found or tree.generation_count >= 5:
        break

print("\nSearch finished!")
if tree.path_found:
    print("SUCCESS.")


Δώσε SMILES για το start molecule: CC(=O)Nc1ccc(O)cc1
Δώσε SMILES για το target molecule: CC(=O)Nc1ccc(OC)cc1
Generation #1
Molecules in tree: 8
Closest to target: CC(=O)NC1=CC=C(OS(=O)(=O)O)C=C1 (Distance: 0.3529)
----------------------------------------

Search finished!
SUCCESS.


In [2]:
import random
from rdkit import Chem
from molpher.core import MolpherMol, MolpherAtom
from molpher.core.morphing.operators import MorphingOperator
from rdkit.Chem.EnumerateStereoisomers import EnumerateStereoisomers, StereoEnumerationOptions
from rdkit.Chem import rdChemReactions
from rdkit.Chem import rdmolops
from rdkit.Chem import Descriptors  
from molpher.core import ExplorationTree as ETree


class OxidizeAlcohol(MorphingOperator):
    def __init__(self):
        super(OxidizeAlcohol, self).__init__()
        self._name = "Oxidize Alcohol"
        self._target_atoms = [] 
        self.PATTERN = Chem.MolFromSmarts("[OX2H][#6X4;H1,H2]")

    def setOriginal(self, mol):
        super(OxidizeAlcohol, self).setOriginal(mol)
        self._target_atoms = []
        
        if not self.original:
            return
            
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: 
            return
            
        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
        
            self._target_atoms.append((match[0], match[1]))

    def morph(self):
        if not self.original: 
            return None
            
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: 
            return None
            
        if not self._target_atoms:
            return MolpherMol(other=rdkit_mol)
            
        idx_o, idx_c = random.choice(self._target_atoms)
        
        try:
            rw_mol = Chem.RWMol(rdkit_mol)
            
            if rw_mol.GetBondBetweenAtoms(idx_o, idx_c):
                rw_mol.RemoveBond(idx_o, idx_c)
            rw_mol.AddBond(idx_o, idx_c, Chem.BondType.DOUBLE)
            
            new_mol = rw_mol.GetMol()
    
            for idx in [idx_o, idx_c]:
                atom = new_mol.GetAtomWithIdx(idx)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
            
        except Exception as e:
            return MolpherMol(other=rdkit_mol)

    def getName(self):
        return self._name


class OxidizeAldehydeToAcid(MorphingOperator):
    def __init__(self):
        super(OxidizeAldehydeToAcid, self).__init__()
        self._name = "Oxidize Aldehyde to Acid"
        self._target_carbons = [] 
        self.PATTERN = Chem.MolFromSmarts("[CX3H1](=O)[#6,#1]")

    def setOriginal(self, mol):
        super(OxidizeAldehydeToAcid, self).setOriginal(mol)
        self._target_carbons = []
        
        if not self.original:
            return
            
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: 
            return
            
        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            self._target_carbons.append(match[0])

    def morph(self):
        if not self.original: 
            return None
            
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: 
            return None
        
        if not self._target_carbons:
            return MolpherMol(other=rdkit_mol)
            
        idx_c = random.choice(self._target_carbons)
        
        try:
            rw_mol = Chem.RWMol(rdkit_mol)
            
            new_o_idx = rw_mol.AddAtom(Chem.Atom(8))
            
            rw_mol.AddBond(idx_c, new_o_idx, Chem.BondType.SINGLE)
            
            new_mol = rw_mol.GetMol()
            
            for idx in [idx_c, new_o_idx]:
                atom = new_mol.GetAtomWithIdx(idx)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
        
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
            
        except Exception as e:
            return MolpherMol(other=rdkit_mol)
        
    def getName(self):
        return self._name


class AlkeneToAlcohol(MorphingOperator):
    def __init__(self):
        super(AlkeneToAlcohol, self).__init__()
        self._name = "Markovnikov Hydration"
        self._target_bonds = [] 
        self.PATTERN = Chem.MolFromSmarts("[CX3;H1,H2]=[CX3;H0,H1,H2]")

    def setOriginal(self, mol):
        super(AlkeneToAlcohol, self).setOriginal(mol)
        self._target_bonds = []
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return
        
        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            bond = rdkit_mol.GetBondBetweenAtoms(match[0], match[1])
            if bond and not bond.GetIsAromatic():
                self._target_bonds.append((match[0], match[1]))

    def morph(self):
        if not self.original: return None
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return None
        if not self._target_bonds: return MolpherMol(other=rdkit_mol)
            
        idx1, idx2 = random.choice(self._target_bonds)
        
        try:
            rw_mol = Chem.RWMol(rdkit_mol)
            
            bond = rw_mol.GetBondBetweenAtoms(idx1, idx2)
            if bond:
                bond.SetBondType(Chem.BondType.SINGLE)
            
            h1 = rw_mol.GetAtomWithIdx(idx1).GetTotalNumHs()
            h2 = rw_mol.GetAtomWithIdx(idx2).GetTotalNumHs()
            idx_with_oh = idx1 if h1 <= h2 else idx2
            
            oh_idx = rw_mol.AddAtom(Chem.Atom(8))
            rw_mol.AddBond(idx_with_oh, oh_idx, Chem.BondType.SINGLE)
            
            new_mol = rw_mol.GetMol()
            for idx in [idx1, idx2, oh_idx]:
                atom = new_mol.GetAtomWithIdx(idx)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            return MolpherMol(other=new_mol)
        except:
            return MolpherMol(other=rdkit_mol)
    
    def getName(self): return self._name


class AlcoholToAlkene(MorphingOperator):
    def __init__(self):
        super(AlcoholToAlkene, self).__init__()
        self._name = "Saytzeff Dehydration"
        self._target_groups = [] 
        self.PATTERN = Chem.MolFromSmarts("[OX2H][#6X4;H1,H2;!$(C(O)=O)]")

    def setOriginal(self, mol):
        super(AlcoholToAlkene, self).setOriginal(mol)
        self._target_groups = []
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return
        
        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            self._target_groups.append((match[0], match[1]))

    def morph(self):
        if not self.original: return None
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return None
        if not self._target_groups: return MolpherMol(other=rdkit_mol)
            
        oh_idx, alpha_idx = random.choice(self._target_groups)
        alpha_atom = rdkit_mol.GetAtomWithIdx(alpha_idx)
        
        beta_carbons = [a for a in alpha_atom.GetNeighbors() if a.GetAtomicNum() == 6 and a.GetHybridization() == Chem.HybridizationType.SP3]
        if not beta_carbons: return MolpherMol(other=rdkit_mol)

        beta_carbons.sort(key=lambda x: x.GetTotalNumHs())
        beta_idx = beta_carbons[0].GetIdx()
        
        try:
            rw_mol = Chem.RWMol(rdkit_mol)
            
            bond_ab = rw_mol.GetBondBetweenAtoms(alpha_idx, beta_idx)
            if bond_ab:
                bond_ab.SetBondType(Chem.BondType.DOUBLE)
                
            bond_oh = rw_mol.GetBondBetweenAtoms(oh_idx, alpha_idx)
            if bond_oh:
                rw_mol.RemoveBond(oh_idx, alpha_idx)
            
            new_mol = rw_mol.GetMol()
            
            for idx in [alpha_idx, beta_idx]:
                atom = new_mol.GetAtomWithIdx(idx)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
            
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            
            frags = Chem.GetMolFrags(new_mol, asMols=True)
            if frags:

                frags = sorted(frags, key=lambda x: x.GetNumAtoms(), reverse=True)
                final_mol = frags[0]
                
                clean_smiles = Chem.MolToSmiles(final_mol)
                return MolpherMol(clean_smiles)
                
            return MolpherMol(other=rdkit_mol)
        except:
            return MolpherMol(other=rdkit_mol)
    
    def getName(self): return self._name


class HeteroatomOxidation(MorphingOperator):
    def __init__(self):
        super(HeteroatomOxidation, self).__init__()
        self._name = "Heteroatom Oxidation (Phase I)"
        self._matches = []
        self.N_PATTERN = Chem.MolFromSmarts("[N;X3;H0;!$(N-C=O);!a](-[#6])-[#6]")
        self.S_THIOETHER = Chem.MolFromSmarts("[S;X2;H0;!a]")
        self.S_SULFOXIDE = Chem.MolFromSmarts("[S;X3;D3;H0](=O)")

    def setOriginal(self, mol):
        super(HeteroatomOxidation, self).setOriginal(mol)
        self._matches = []
        
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return
        
        if self.N_PATTERN is not None:
            for match in rdkit_mol.GetSubstructMatches(self.N_PATTERN):
                self._matches.append((match[0], "N"))

        if self.S_THIOETHER is not None:
            for match in rdkit_mol.GetSubstructMatches(self.S_THIOETHER):
                self._matches.append((match[0], "S_thio"))

        if self.S_SULFOXIDE is not None:
            for match in rdkit_mol.GetSubstructMatches(self.S_SULFOXIDE):
                self._matches.append((match[0], "S_sulf"))

    def morph(self):
        if not self.original: return None
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return None

        if not self._matches:
            return MolpherMol(other=rdkit_mol)

        target_idx, atom_type = random.choice(self._matches)

        try:
            rw_mol = Chem.RWMol(rdkit_mol)
            
            oxygen_idx = rw_mol.AddAtom(Chem.Atom(8))
            
            target_atom = rw_mol.GetAtomWithIdx(target_idx)
            ox_atom = rw_mol.GetAtomWithIdx(oxygen_idx)

            if atom_type in ["S_thio", "S_sulf"]:

                rw_mol.AddBond(target_idx, oxygen_idx, Chem.BondType.DOUBLE)
                
                target_atom.SetNoImplicit(False)
                target_atom.SetNumExplicitHs(0)
                
            elif atom_type == "N":
                
                rw_mol.AddBond(target_idx, oxygen_idx, Chem.BondType.SINGLE)
                target_atom.SetFormalCharge(1)
                ox_atom.SetFormalCharge(-1)
                ox_atom.SetNoImplicit(True)
                ox_atom.SetNumExplicitHs(0)
                target_atom.SetNoImplicit(False)
                target_atom.SetNumExplicitHs(0)

            new_mol = rw_mol.GetMol()
            
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self):
        return self._name


class HydrolyzeEster(MorphingOperator):
    def __init__(self):
        super(HydrolyzeEster, self).__init__()
        self._name = "Ester Hydrolysis (Generalized)"
        self._matches = []
        self.PATTERN = Chem.MolFromSmarts("[CX3](=O)[OX2][#6]")

    def setOriginal(self, mol):
        super(HydrolyzeEster, self).setOriginal(mol)
        self._matches = []

        if not self.original:
            return

        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None:
            return

        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)

        for match in matches:
            carbonyl_c_idx = match[0]   
            carbonyl_o_idx = match[1]  
            ester_o_idx = match[2]      
            alkoxy_c_idx = match[3]   

            carbonyl_c = rdkit_mol.GetAtomWithIdx(carbonyl_c_idx)
            alkoxy_atom = rdkit_mol.GetAtomWithIdx(alkoxy_c_idx)
            
            oxygen_neighbors = [
                nb for nb in carbonyl_c.GetNeighbors()
                if nb.GetAtomicNum() == 8
            ]
            if len(oxygen_neighbors) > 2:
                continue

            # EXCLUSION: tert-butyl esters
            
            carbon_neighbors = [
                nb for nb in alkoxy_atom.GetNeighbors()
                if nb.GetAtomicNum() == 6
            ]
            if len(carbon_neighbors) == 3:
                continue

            self._matches.append(
                (carbonyl_c_idx, carbonyl_o_idx, ester_o_idx, alkoxy_c_idx)
            )

    def morph(self):
        if not self.original:
            return None

        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None:
            return None

        if not self._matches:
            
            return MolpherMol(other=rdkit_mol)

        carbonyl_c_idx, carbonyl_o_idx, ester_o_idx, alkoxy_c_idx = random.choice(self._matches)

        try:
            rw_mol = Chem.RWMol(rdkit_mol)

            if rw_mol.GetBondBetweenAtoms(carbonyl_c_idx, ester_o_idx):
                rw_mol.RemoveBond(carbonyl_c_idx, ester_o_idx)
            else:
                return MolpherMol(other=rdkit_mol)

            new_oh_idx = rw_mol.AddAtom(Chem.Atom(8))
            rw_mol.AddBond(carbonyl_c_idx, new_oh_idx, Chem.BondType.SINGLE)
            
            new_mol = rw_mol.GetMol()

            fragments = Chem.GetMolFrags(new_mol, asMols=True, sanitizeFrags=False)
            if not fragments:
                return MolpherMol(other=rdkit_mol)

            processed_frags = []
            for frag in fragments:
                frag_rw = Chem.RWMol(frag)
                for atom in frag_rw.GetAtoms():
                    atom.SetNoImplicit(False)
                    atom.SetNumExplicitHs(0)

                frag_mol = frag_rw.GetMol()
                try:
                    frag_mol.UpdatePropertyCache(strict=False)
                    Chem.SanitizeMol(frag_mol)
                    processed_frags.append(frag_mol)
                except Exception:
                    continue

            if not processed_frags:
                return MolpherMol(other=rdkit_mol)

            benzene = Chem.MolFromSmarts("c1ccccc1")
            ring_fragments = [f for f in processed_frags if f.HasSubstructMatch(benzene)]

            if ring_fragments:
                largest_frag = max(ring_fragments, key=lambda m: Descriptors.MolWt(m))
            else:
                largest_frag = max(processed_frags, key=lambda m: Descriptors.MolWt(m))

            largest_frag.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(largest_frag, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(largest_frag, cleanIt=True, force=True)

            return MolpherMol(other=largest_frag)

        except Exception as e:
            print(f"[Debug Error]: {e}")
            return MolpherMol(other=rdkit_mol)

    def getName(self):
        return self._name


class NDealkylation(MorphingOperator):
    def __init__(self):
        super(NDealkylation, self).__init__()
        self._name = "N-Dealkylation (Phase I - Advanced)"
        self._target_bonds = []
        self.PATTERN = Chem.MolFromSmarts("[NX3;H0,H1;!a;!$(N-C=O)][CX4;H1,H2,H3]")

    def setOriginal(self, mol):
        super(NDealkylation, self).setOriginal(mol)
        self._target_bonds = []
        
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return
        
        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            self._target_bonds.append((match[0], match[1]))

    def morph(self):
        if not self.original: return None
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return None
        
        if not self._target_bonds:
            return MolpherMol(other=rdkit_mol)
            
        idx_n, idx_c = random.choice(self._target_bonds)
        
        try:
            rw_mol = Chem.RWMol(rdkit_mol)
            
            bond = rw_mol.GetBondBetweenAtoms(idx_n, idx_c)
            if bond:
                rw_mol.RemoveBond(idx_n, idx_c)
                
            new_mol = rw_mol.GetMol()
            
            atom_n = new_mol.GetAtomWithIdx(idx_n)
            atom_n.SetNoImplicit(False)
            atom_n.SetNumExplicitHs(0)
            
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            
            frags_mols = Chem.GetMolFrags(new_mol, asMols=True)
            
            if frags_mols:
                
                frags_indices = Chem.GetMolFrags(new_mol, asMols=False)
                frags_with_meta = list(zip(frags_mols, frags_indices))
                frags_with_meta = sorted(
                    frags_with_meta, 
                    key=lambda x: (idx_n in x[1], x[0].GetNumAtoms()), 
                    reverse=True
                )
                
                final_mol = frags_with_meta[0][0]
                
                Chem.AssignStereochemistry(final_mol, cleanIt=True, force=True)
                clean_smiles = Chem.MolToSmiles(final_mol)
                return MolpherMol(clean_smiles)
                
            return MolpherMol(other=rdkit_mol)
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name


class AromaticHydroxylation(MorphingOperator):
    def __init__(self):
        super(AromaticHydroxylation, self).__init__()
        self._name = "Aromatic Hydroxylation (Phase I - Regioselective)"
        self._matches = []
        self.AROMATIC_C = Chem.MolFromSmarts("[c;H1]")

    def _get_para_score(self, mol, c_idx):
        """ Υπολογισμός para-θέσης σε 6μελείς δακτυλίους """
        for ring in mol.GetRingInfo().AtomRings():
            if c_idx in ring and len(ring) == 6:
                for r_idx in ring:
                    r_atom = mol.GetAtomWithIdx(r_idx)
                    has_ex_neighbor = any(n.GetIdx() not in ring for n in r_atom.GetNeighbors())
                    
                    if has_ex_neighbor:
                        path = Chem.GetShortestPath(mol, r_idx, c_idx)
                        if len(path) == 4: # Απόσταση para
                            return 10
        return 1 

    def setOriginal(self, mol):
        super(AromaticHydroxylation, self).setOriginal(mol)
        self._matches = []
        
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return

        if self.AROMATIC_C is not None:
            matches = rdkit_mol.GetSubstructMatches(self.AROMATIC_C)
            scored_sites = []
            for match in matches:
                c_idx = match[0]
                score = self._get_para_score(rdkit_mol, c_idx)
                scored_sites.append((c_idx, score))
                
            if scored_sites:
                max_score = max(site[1] for site in scored_sites)
                self._matches = [site[0] for site in scored_sites if site[1] == max_score]

    def morph(self):
        if not self.original: return None
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return None

        if not self._matches:
            return MolpherMol(other=rdkit_mol)

        c_idx = random.choice(self._matches)
        
        try:
            rw_mol = Chem.RWMol(rdkit_mol)
            
            new_o_idx = rw_mol.AddAtom(Chem.Atom(8))
            c_atom = rw_mol.GetAtomWithIdx(c_idx)
            o_atom = rw_mol.GetAtomWithIdx(new_o_idx)
            
            c_atom.SetNoImplicit(False)
            c_atom.SetNumExplicitHs(0)
            o_atom.SetNoImplicit(False)
            o_atom.SetNumExplicitHs(0)
            
            rw_mol.AddBond(c_idx, new_o_idx, Chem.BondType.SINGLE)
            
            new_mol = rw_mol.GetMol()
            
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name

    
class AliphaticEpoxidation(MorphingOperator):
    def __init__(self):
        super(AliphaticEpoxidation, self).__init__()
        self._name = "Aliphatic Epoxidation (Phase I - Stereospecific)"
        self._target_bonds = []
        self.DOUBLE_BOND_PATTERN = Chem.MolFromSmarts("[CX3;!a]=[CX3;!a]")

    def setOriginal(self, mol):
        super(AliphaticEpoxidation, self).setOriginal(mol)
        self._target_bonds = []
        
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return

        if self.DOUBLE_BOND_PATTERN is not None:
            matches = rdkit_mol.GetSubstructMatches(self.DOUBLE_BOND_PATTERN)
            for match in matches:
                
                pair = tuple(sorted([match[0], match[1]]))
                if pair not in self._target_bonds:
                    self._target_bonds.append(pair)

    def morph(self):
        if not self.original: return None
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return None

        if not self._target_bonds:
            return MolpherMol(other=rdkit_mol)
        
        idx1, idx2 = random.choice(self._target_bonds)

        try:
            rw_mol = Chem.RWMol(rdkit_mol)
            
            bond = rw_mol.GetBondBetweenAtoms(idx1, idx2)
            if bond:
                bond.SetBondType(Chem.BondType.SINGLE)
            
            oxygen_idx = rw_mol.AddAtom(Chem.Atom(8))
            
            rw_mol.AddBond(idx1, oxygen_idx, Chem.BondType.SINGLE)
            rw_mol.AddBond(idx2, oxygen_idx, Chem.BondType.SINGLE)
            
            new_mol = rw_mol.GetMol()

            for idx in [idx1, idx2, oxygen_idx]:
                atom = new_mol.GetAtomWithIdx(idx)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
    
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self):
        return self._name


class AzoReduction(MorphingOperator):
    def __init__(self):
        super(AzoReduction, self).__init__()
        self._name = "Azo Reduction (Phase I - Safe Fragmentation)"
        self._matches = []
        self.AZO_PATTERN = Chem.MolFromSmarts("[C,c]-[N;!R]=[N;!R]-[C,c]")

    def setOriginal(self, mol):
        super(AzoReduction, self).setOriginal(mol)
        self._matches = []

        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return

        matches = rdkit_mol.GetSubstructMatches(self.AZO_PATTERN)
        for match in matches:
            self._matches.append((match[0], match[1], match[2], match[3]))

    def morph(self):
        if not self.original: return None
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return None

        if not self._matches:
            return MolpherMol(other=rdkit_mol)

        edit_mol = Chem.RWMol(rdkit_mol)
        c1_idx, n1_idx, n2_idx, c2_idx = random.choice(self._matches)

        try:
            edit_mol.RemoveBond(n1_idx, n2_idx)

            for idx in [n1_idx, n2_idx]:
                atom = edit_mol.GetAtomWithIdx(idx)
                atom.SetFormalCharge(0)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                for prop in list(atom.GetPropNames()):
                    atom.ClearProp(prop)

            new_mol = edit_mol.GetMol()
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            
            fragments = rdmolops.GetMolFrags(new_mol, asMols=True)
            if not fragments:
                return MolpherMol(other=rdkit_mol)

            chosen_frag = random.choice(fragments)
            
            Chem.AssignStereochemistry(chosen_frag, cleanIt=True, force=True)
            return MolpherMol(other=chosen_frag)
                
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name

    
class NitroReduction(MorphingOperator):
    def __init__(self):
        super(NitroReduction, self).__init__()
        self._name = "Nitro Reduction (Phase I - Safe)"
        self._target_nitrogens = []
        self.NITRO_PATTERN = Chem.MolFromSmarts("[C,c][N;X3](=[O,O-])~[O,O-]")

    def setOriginal(self, mol):
        super(NitroReduction, self).setOriginal(mol)
        self._target_nitrogens = []

        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return

        matches = rdkit_mol.GetSubstructMatches(self.NITRO_PATTERN)
        for match in matches:
            if match[1] not in self._target_nitrogens:
                self._target_nitrogens.append(match[1])

    def morph(self):
        if not self.original: return None
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return None

        if not self._target_nitrogens:
            return MolpherMol(other=rdkit_mol)

        n_idx = random.choice(self._target_nitrogens)

        try:
            rw_mol = Chem.RWMol(rdkit_mol)
            n_atom = rw_mol.GetAtomWithIdx(n_idx)
            
            o_indices = [neighbor.GetIdx() for neighbor in n_atom.GetNeighbors() if neighbor.GetAtomicNum() == 8]
            
            for o_idx in o_indices:
                bond = rw_mol.GetBondBetweenAtoms(n_idx, o_idx)
                if bond:
                    rw_mol.RemoveBond(n_idx, o_idx)
            
            n_atom.SetFormalCharge(0)
            n_atom.SetNoImplicit(False)
            n_atom.SetNumExplicitHs(2)
            for prop in list(n_atom.GetPropNames()):
                n_atom.ClearProp(prop)
            
            new_mol = rw_mol.GetMol()
            new_mol.UpdatePropertyCache(strict=False)
            
            atoms_to_remove = []
            for atom in new_mol.GetAtoms():
                if atom.GetAtomicNum() == 8 and atom.GetDegree() == 0:
                    atoms_to_remove.append(atom.GetIdx())

            edit = Chem.RWMol(new_mol)
            for idx in sorted(atoms_to_remove, reverse=True):
                edit.RemoveAtom(idx)
            new_mol = edit.GetMol()

            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            return MolpherMol(other=new_mol)

        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name


class AldehydeReduction(MorphingOperator):
    def __init__(self):
        super(AldehydeReduction, self).__init__()
        self._name = "Aldehyde Reduction (Phase I - Safe)"
        self._matches = []
        self.ALDEHYDE_PATTERN = Chem.MolFromSmarts("[CX3H1;!$(C(=O)[O,N,S,F,Cl,Br,I])]=[OX1]")

    def setOriginal(self, mol):
        super(AldehydeReduction, self).setOriginal(mol)
        self._matches = []

        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return

        matches = rdkit_mol.GetSubstructMatches(self.ALDEHYDE_PATTERN)
        for match in matches:
            self._matches.append((match[0], match[1]))

    def morph(self):
        if not self.original: return None
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return None

        if not self._matches:
            return MolpherMol(other=rdkit_mol)

        edit_mol = Chem.RWMol(rdkit_mol)
        c_idx, o_idx = random.choice(self._matches)

        try:
            bond = edit_mol.GetBondBetweenAtoms(c_idx, o_idx)
            if bond is None: return MolpherMol(other=rdkit_mol)

            bond.SetBondType(Chem.BondType.SINGLE)

            for idx in [c_idx, o_idx]:
                atom = edit_mol.GetAtomWithIdx(idx)
                atom.SetFormalCharge(0)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                for prop in list(atom.GetPropNames()):
                    atom.ClearProp(prop)
                atom.UpdatePropertyCache(strict=False)

            new_mol = edit_mol.GetMol()
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name


class KetoneReduction(MorphingOperator):
    def __init__(self):
        super(KetoneReduction, self).__init__()
        self._name = "Ketone Reduction (Phase I - Stereospecific Enumerate)"
        self._matches = []

    def setOriginal(self, mol):
        super(KetoneReduction, self).setOriginal(mol)
        self._matches = []

        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return
        
        ketone_pattern = Chem.MolFromSmarts("[CX3;$(C(=O)(-[#6])-[#6])]=O")
        matches = rdkit_mol.GetSubstructMatches(ketone_pattern)
        for match in matches:
            self._matches.append((match[0], match[1]))

    def morph(self):
        if not self.original: return None
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return None

        if not self._matches:
            return MolpherMol(other=rdkit_mol)

        edit_mol = Chem.RWMol(rdkit_mol)
        carbonyl_idx, oxygen_idx = random.choice(self._matches)

        try:
            bond = edit_mol.GetBondBetweenAtoms(carbonyl_idx, oxygen_idx)
            if bond is None: return MolpherMol(other=rdkit_mol)

            bond.SetBondType(Chem.BondType.SINGLE)

            for idx in [carbonyl_idx, oxygen_idx]:
                atom = edit_mol.GetAtomWithIdx(idx)
                atom.SetFormalCharge(0)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                atom.SetChiralTag(Chem.ChiralType.CHI_UNSPECIFIED)
                if atom.HasProp('_CIPCode'): atom.ClearProp('_CIPCode')
                for prop in list(atom.GetPropNames()): atom.ClearProp(prop)
                atom.UpdatePropertyCache(strict=False)

            new_mol = edit_mol.GetMol()
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)

            options = StereoEnumerationOptions(onlyUnassigned=True)
            isomers = list(EnumerateStereoisomers(new_mol, options=options))

            if isomers:
                chosen_iso = random.choice(isomers)
                Chem.SanitizeMol(chosen_iso)
                Chem.AssignStereochemistry(chosen_iso, cleanIt=True, force=True)
                return MolpherMol(other=chosen_iso)

            return MolpherMol(other=new_mol)
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name


class AlkenylReduction(MorphingOperator):
    def __init__(self):
        super(AlkenylReduction, self).__init__()
        self._name = "Alkenyl Reduction (Phase I - Ring Safe)"
        self._target_bonds = []
        self.PATTERN = Chem.MolFromSmarts("[CX3;!a]=[CX3;!a]")

    def setOriginal(self, mol):
        super(AlkenylReduction, self).setOriginal(mol)
        self._target_bonds = []
        
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return

        if self.PATTERN is not None:
            matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
            for match in matches:
                pair = tuple(sorted([match[0], match[1]]))
                if pair not in self._target_bonds:
                    self._target_bonds.append(pair)

    def morph(self):
        if not self.original: return None
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return None

        if not self._target_bonds:
            return MolpherMol(other=rdkit_mol)
        
        idx1, idx2 = random.choice(self._target_bonds)

        try:
            rw_mol = Chem.RWMol(rdkit_mol)
            
            bond = rw_mol.GetBondBetweenAtoms(idx1, idx2)
            if bond:
                bond.SetBondType(Chem.BondType.SINGLE)
                bond.SetStereo(Chem.BondStereo.STEREONONE)
            
            for idx in [idx1, idx2]:
                atom = rw_mol.GetAtomWithIdx(idx)
                atom.SetFormalCharge(0)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                atom.SetChiralTag(Chem.ChiralType.CHI_UNSPECIFIED)
                if atom.HasProp('_CIPCode'): 
                    atom.ClearProp('_CIPCode')
                for prop in list(atom.GetPropNames()):
                    atom.ClearProp(prop)
                atom.UpdatePropertyCache(strict=False)
            
            new_mol = rw_mol.GetMol()
            
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self):
        return self._name


class LactamFormation(MorphingOperator):
    def __init__(self):
        super(LactamFormation, self).__init__()
        self._name = "Lactam Formation (Phase I - Ring Oxidation Safe)"
        self._matches = []
        self.PATTERN = Chem.MolFromSmarts("[NX3;R;!$(N-C=O);!a]-[CX4;R;H2;!$(C-O)]")

    def setOriginal(self, mol):
        super(LactamFormation, self).setOriginal(mol)
        self._matches = []

        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return

        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            self._matches.append((match[0], match[1]))

    def morph(self):
        if not self.original: return None
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return None

        if not self._matches:
            return MolpherMol(other=rdkit_mol)

        n_idx, c_idx = random.choice(self._matches)

        try:
            rw_mol = Chem.RWMol(rdkit_mol)
            
            o_atom = Chem.Atom(8)
            o_atom.SetFormalCharge(0)
            o_idx = rw_mol.AddAtom(o_atom)
            
            rw_mol.AddBond(c_idx, o_idx, Chem.BondType.DOUBLE)
            
            for idx in [n_idx, c_idx, o_idx]:
                atom = rw_mol.GetAtomWithIdx(idx)
                atom.SetFormalCharge(0)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                for prop in list(atom.GetPropNames()):
                    atom.ClearProp(prop)
                atom.UpdatePropertyCache(strict=False)
            
            new_mol = rw_mol.GetMol()
            
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self):
        return self._name


class AcylGlucuronidation(MorphingOperator):
    def __init__(self):
        super(AcylGlucuronidation, self).__init__()
        self._name = "Acyl Glucuronidation (Carboxylic Acids)"
        self._target_oxygens = []
        self.PATTERN = Chem.MolFromSmarts("[CX3](=O)[OX2H]")
        self.GLUCURONIDE_TEMPLATE = Chem.MolFromSmiles("C1(O)O[C@H](C(=O)O)[C@@H](O)[C@H](O)[C@@H]1O")

    def setOriginal(self, mol):
        super(AcylGlucuronidation, self).setOriginal(mol)
        self._target_oxygens = []
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return
        
        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            self._target_oxygens.append(match[2])

    def morph(self):
        if not self.original or not self._target_oxygens:
            return MolpherMol(other=self.original.asRDMol())
        
        rdkit_mol = self.original.asRDMol()
        try:
            target_o_idx = random.choice(self._target_oxygens)
            
            combined = Chem.CombineMols(rdkit_mol, self.GLUCURONIDE_TEMPLATE)
            rw_combined = Chem.RWMol(combined)
            
            c_sugar_idx = rdkit_mol.GetNumAtoms()
            oh_sugar_idx = c_sugar_idx + 1
            
            rw_combined.AddBond(target_o_idx, c_sugar_idx, Chem.BondType.SINGLE)
        
            rw_combined.RemoveAtom(oh_sugar_idx)
        
            new_mol = rw_combined.GetMol()
            
            for idx in [target_o_idx, c_sugar_idx]:
                atom = new_mol.GetAtomWithIdx(idx)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
            
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name

   
class AlcoholPhenolGlucuronidation(MorphingOperator):
    def __init__(self):
        super(AlcoholPhenolGlucuronidation, self).__init__()
        self._name = "O-Glucuronidation (Alcohols/Phenols)"
        self._target_oxygens = []
        
        # SMARTS: Επιλέγει [OX2H] που συνδέεται με άνθρακα, ο οποίος ΔΕΝ είναι καρβονύλιο
        self.PATTERN = Chem.MolFromSmarts("[#6;!$(C=O);!$(C=C)][OX2H]")
        # Template: β-D-glucuronide με SMILES όπου ο C1 (ανωμερής) είναι το 1ο άτομο (index 0)
        # SMILES: C1([OH])O[C@H](C(=O)O)[C@@H](O)[C@H](O)[C@@H]1O
        # Με αυτό το SMILES: 
        # index 0 -> ο C1 που θα ενωθεί με το υπόλοιπο μόριο
        # index 1 -> το -OH του C1 το οποίο ΠΡΕΠΕΙ να αφαιρεθεί
        self.GLUCURONIDE_TEMPLATE = Chem.MolFromSmiles("C1(O)O[C@H](C(=O)O)[C@@H](O)[C@H](O)[C@@H]1O")

    def setOriginal(self, mol):
        super(AlcoholPhenolGlucuronidation, self).setOriginal(mol)
        self._target_oxygens = []
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return

        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            # Το [OX2H] είναι το δεύτερο άτομο στο pattern (index 1)
            self._target_oxygens.append(match[1])

    def morph(self):
        if not self.original or not self._target_oxygens:
            return MolpherMol(other=self.original.asRDMol())

        rdkit_mol = self.original.asRDMol()
        try:
            target_o_idx = random.choice(self._target_oxygens)
            
            combined = Chem.CombineMols(rdkit_mol, self.GLUCURONIDE_TEMPLATE)
            rw_combined = Chem.RWMol(combined)
            c_sugar_idx = rdkit_mol.GetNumAtoms() 
            oh_sugar_idx = c_sugar_idx + 1 
            
            rw_combined.AddBond(target_o_idx, c_sugar_idx, Chem.BondType.SINGLE)
            rw_combined.RemoveAtom(oh_sugar_idx)
            
            new_mol = rw_combined.GetMol()
            
            for idx in [target_o_idx, c_sugar_idx]:
                atom = new_mol.GetAtomWithIdx(idx)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
            
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name


    
class AlcoholPhenolSulfation(MorphingOperator):
    def __init__(self):
        super(AlcoholPhenolSulfation, self).__init__()
        self._name = "O-Sulfation (Alcohols/Phenols)"
        self._target_atoms = []
        self.PATTERN = Chem.MolFromSmarts("[OX2H][#6;!$(C=O)]")

    def setOriginal(self, mol):
        super(AlcoholPhenolSulfation, self).setOriginal(mol)
        self._target_atoms = []

        if not self.original:
            return

        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None:
            return

        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            self._target_atoms.append(match[0])

    def morph(self):
        if not self.original:
            return None

        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None:
            return None

        if not self._target_atoms:
            return MolpherMol(other=rdkit_mol)

        oxygen_idx = random.choice(self._target_atoms)

        try:
            rw_mol = Chem.RWMol(rdkit_mol)

            # Εισαγωγή ουδέτερης ομάδας -SO3H
            sulfur_idx = rw_mol.AddAtom(Chem.Atom(16)) # S
            rw_mol.AddBond(oxygen_idx, sulfur_idx, Chem.BondType.SINGLE)

            o1_idx = rw_mol.AddAtom(Chem.Atom(8)) # =O
            rw_mol.AddBond(sulfur_idx, o1_idx, Chem.BondType.DOUBLE)

            o2_idx = rw_mol.AddAtom(Chem.Atom(8)) # =O
            rw_mol.AddBond(sulfur_idx, o2_idx, Chem.BondType.DOUBLE)

            o3_idx = rw_mol.AddAtom(Chem.Atom(8)) # -OH (Ουδέτερο)
            rw_mol.AddBond(sulfur_idx, o3_idx, Chem.BondType.SINGLE)

            new_mol = rw_mol.GetMol()

            for idx in [oxygen_idx, sulfur_idx, o1_idx, o2_idx, o3_idx]:
                atom = new_mol.GetAtomWithIdx(idx)
                if atom.GetAtomicNum() != 16: # Αφήνουμε το Θείο να διαχειριστεί το σθένος του (6)
                    atom.SetNoImplicit(False)
                    atom.SetNumExplicitHs(0)

            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)

            return MolpherMol(other=new_mol)

        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self):
        return self._name


class NitrogenSulfation(MorphingOperator):
    def __init__(self):
        super(NitrogenSulfation, self).__init__()
        self._name = "N-Sulfation (Amines 1°, 2°, 3° & Aromatic)"
        self._target_atoms = []
        self.PATTERN = Chem.MolFromSmarts("[N;X3;!$(N-C=O);!$(N-C(=O)N);!$(N-C(=O)O)]")

    def setOriginal(self, mol):
        super(NitrogenSulfation, self).setOriginal(mol)
        self._target_atoms = []
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return
        
        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            self._target_atoms.append(match[0])

    def morph(self):
        if not self.original: return None
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return None
        
        if not self._target_atoms:
            return MolpherMol(other=rdkit_mol)
        
        nitrogen_idx = random.choice(self._target_atoms)
        
        try:
            rw_mol = Chem.RWMol(rdkit_mol)
            is_tertiary = (rw_mol.GetAtomWithIdx(nitrogen_idx).GetTotalNumHs() == 0)
            
            sulfur_idx = rw_mol.AddAtom(Chem.Atom(16)) # S
            rw_mol.AddBond(nitrogen_idx, sulfur_idx, Chem.BondType.SINGLE)
            
            o1_idx = rw_mol.AddAtom(Chem.Atom(8)) # =O
            rw_mol.AddBond(sulfur_idx, o1_idx, Chem.BondType.DOUBLE)
            
            o2_idx = rw_mol.AddAtom(Chem.Atom(8)) # =O
            rw_mol.AddBond(sulfur_idx, o2_idx, Chem.BondType.DOUBLE)
            
            o3_idx = rw_mol.AddAtom(Chem.Atom(8)) # -OH
            rw_mol.AddBond(sulfur_idx, o3_idx, Chem.BondType.SINGLE)
            
            new_mol = rw_mol.GetMol()
            
            for idx in [o1_idx, o2_idx, o3_idx]:
                atom = new_mol.GetAtomWithIdx(idx)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                atom.SetFormalCharge(0)
            
            n_atom = new_mol.GetAtomWithIdx(nitrogen_idx)
            n_atom.SetNoImplicit(False)
            n_atom.SetNumExplicitHs(0)
            
            if is_tertiary:
                
                n_atom.SetFormalCharge(1)
            else:
                
                n_atom.SetFormalCharge(0)
                
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
            
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name
    

class AminoAcidConjugation(MorphingOperator):
    def __init__(self):
        super(AminoAcidConjugation, self).__init__()
        self._name = "Amino Acid Conjugation (Gly/Tau/Gln)"
        self._target_atoms = []
        self.PATTERN = Chem.MolFromSmarts("[CX3](=O)[OX2H]") 
        # Σε όλα, το Άζωτο (Ν) που θα επιτεθεί είναι το ΠΡΩΤΟ άτομο (index 0)
        self.TEMPLATES = {
            "Glycine": Chem.MolFromSmiles("NCC(=O)O"),
            "Taurine": Chem.MolFromSmiles("NCCS(=O)(=O)O"),
            "Glutamine": Chem.MolFromSmiles("N[C@@H](CCC(=O)N)C(=O)O") # Διατήρηση στερεοχημείας
        }

    def setOriginal(self, mol):
        super(AminoAcidConjugation, self).setOriginal(mol)
        self._target_atoms = []
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return
        
        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            # Κρατάμε το index του Καρβονυλικού Άνθρακα (index 0) και του -OH (index 2)
            self._target_atoms.append((match[0], match[2]))

    def morph(self):
        if not self.original or not self._target_atoms:
            return MolpherMol(other=self.original.asRDMol())
        
        rdkit_mol = self.original.asRDMol()
        try:
            chosen_c_idx, chosen_oh_idx = random.choice(self._target_atoms)
            template_name = random.choice(list(self.TEMPLATES.keys()))
            template_mol = self.TEMPLATES[template_name]
            combined = Chem.CombineMols(rdkit_mol, template_mol)
            rw_combined = Chem.RWMol(combined)
            n_amino_idx = rdkit_mol.GetNumAtoms()
            rw_combined.AddBond(chosen_c_idx, n_amino_idx, Chem.BondType.SINGLE)
            rw_combined.RemoveAtom(chosen_oh_idx)
            
            new_mol = rw_combined.GetMol()
            
            # Λόγω του RemoveAtom, ο index του n_amino_idx μετατοπίστηκε κατά -1 
            # αν ο chosen_oh_idx ήταν μικρότερος, αλλά για ασφάλεια κάνουμε reset στο στοχευμένο καρβονύλιο 
            # και στο άζωτο σαρώνοντας το γράφημα.
            for atom in new_mol.GetAtoms():
                if atom.GetAtomicNum() in [6, 7]: # Άνθρακας καρβονυλίου και Άζωτο αμιδίου
                    atom.SetNoImplicit(False)
                    atom.SetNumExplicitHs(0)
                    atom.SetFormalCharge(0)
            
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
            
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name

    
class NAcetylation(MorphingOperator):
    def __init__(self):
        super(NAcetylation, self).__init__()
        self._name = "N-acetylation (Primary Amines)"
        self._target_atoms = []
        self.PATTERN = Chem.MolFromSmarts("[N;H2;!$(N-C=O);!$(N-S(=O)=O);!$(NN)]")

    def setOriginal(self, mol):
        super(NAcetylation, self).setOriginal(mol)
        self._target_atoms = []
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return
        
        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            self._target_atoms.append(match[0])

    def morph(self):
        if not self.original or not self._target_atoms:
            return MolpherMol(other=self.original.asRDMol())
        
        rdkit_mol = self.original.asRDMol()
        try:
            chosen_idx = random.choice(self._target_atoms)
            rw_mol = Chem.RWMol(rdkit_mol)
            
            c_carbonyl_idx = rw_mol.AddAtom(Chem.Atom(6))
            rw_mol.AddBond(chosen_idx, c_carbonyl_idx, Chem.BondType.SINGLE)
            
            o_carbonyl_idx = rw_mol.AddAtom(Chem.Atom(8))
            rw_mol.AddBond(c_carbonyl_idx, o_carbonyl_idx, Chem.BondType.DOUBLE)
            
            c_methyl_idx = rw_mol.AddAtom(Chem.Atom(6))
            rw_mol.AddBond(c_carbonyl_idx, c_methyl_idx, Chem.BondType.SINGLE)
            
            new_mol = rw_mol.GetMol()
            
            for idx in [chosen_idx, c_carbonyl_idx, o_carbonyl_idx, c_methyl_idx]:
                atom = new_mol.GetAtomWithIdx(idx)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                atom.SetFormalCharge(0)
                
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
            
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name


class HydrazineAcetylation(MorphingOperator):
    def __init__(self):
        super(HydrazineAcetylation, self).__init__()
        self._name = "Hydrazine acetylation"
        self._target_atoms = []
        self.PATTERN = Chem.MolFromSmarts("[NX3;!$(N-C=O)][NX3;H2;!$(N-C=O)]")

    def setOriginal(self, mol):
        super(HydrazineAcetylation, self).setOriginal(mol)
        self._target_atoms = []
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return
        
        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            # Κρατάμε ΜΟΝΟ το index του ακραίου αζώτου [-NH2] 
            self._target_atoms.append(match[1])

    def morph(self):
        if not self.original or not self._target_atoms:
            return MolpherMol(other=self.original.asRDMol())
        
        rdkit_mol = self.original.asRDMol()
        try:
            terminal_n_idx = random.choice(self._target_atoms)
            rw_mol = Chem.RWMol(rdkit_mol)
            
            c_carbonyl_idx = rw_mol.AddAtom(Chem.Atom(6))
            rw_mol.AddBond(terminal_n_idx, c_carbonyl_idx, Chem.BondType.SINGLE)
            
            o_carbonyl_idx = rw_mol.AddAtom(Chem.Atom(8))
            rw_mol.AddBond(c_carbonyl_idx, o_carbonyl_idx, Chem.BondType.DOUBLE)
            
            c_methyl_idx = rw_mol.AddAtom(Chem.Atom(6))
            rw_mol.AddBond(c_carbonyl_idx, c_methyl_idx, Chem.BondType.SINGLE)
            
            new_mol = rw_mol.GetMol()
            
            for idx in [terminal_n_idx, c_carbonyl_idx, o_carbonyl_idx, c_methyl_idx]:
                atom = new_mol.GetAtomWithIdx(idx)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                atom.SetFormalCharge(0)
                
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
            
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name

  
class HydrazideAcetylation(MorphingOperator):
    def __init__(self):
        super(HydrazideAcetylation, self).__init__()
        self._name = "Hydrazide acetylation"
        self._target_atoms = []
        self.PATTERN = Chem.MolFromSmarts("[CX3](=O)[NX3;H1][NX3;H2;!$(N-C=O)]")

    def setOriginal(self, mol):
        super(HydrazideAcetylation, self).setOriginal(mol)
        self._target_atoms = []
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return
        
        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            # Κρατάμε το index του ακραίου αζώτου [-NH2] (index 3)
            self._target_atoms.append(match[3])

    def morph(self):
        if not self.original or not self._target_atoms:
            return MolpherMol(other=self.original.asRDMol())
        
        rdkit_mol = self.original.asRDMol()
        try:
            terminal_n_idx = random.choice(self._target_atoms)
            rw_mol = Chem.RWMol(rdkit_mol)
            
            c_carbonyl_idx = rw_mol.AddAtom(Chem.Atom(6))
            rw_mol.AddBond(terminal_n_idx, c_carbonyl_idx, Chem.BondType.SINGLE)
            
            o_carbonyl_idx = rw_mol.AddAtom(Chem.Atom(8))
            rw_mol.AddBond(c_carbonyl_idx, o_carbonyl_idx, Chem.BondType.DOUBLE)
            
            c_methyl_idx = rw_mol.AddAtom(Chem.Atom(6))
            rw_mol.AddBond(c_carbonyl_idx, c_methyl_idx, Chem.BondType.SINGLE)
            
            new_mol = rw_mol.GetMol()
 
            for idx in [terminal_n_idx, c_carbonyl_idx, o_carbonyl_idx, c_methyl_idx]:
                atom = new_mol.GetAtomWithIdx(idx)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                atom.SetFormalCharge(0)
                
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
            
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name
    

class OMethylation(MorphingOperator):
    def __init__(self):
        super(OMethylation, self).__init__()
        self._name = "O-methylation (Alcohols/Phenols)"
        self._target_atoms = []
        self.PATTERN = Chem.MolFromSmarts("[OX2H][#6;!$(C=O);!$(C=C);!$(C=N)]")

    def setOriginal(self, mol):
        super(OMethylation, self).setOriginal(mol)
        self._target_atoms = []
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return
        
        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            self._target_atoms.append(match[0])

    def morph(self):
        if not self.original or not self._target_atoms:
            return MolpherMol(other=self.original.asRDMol())
        
        rdkit_mol = self.original.asRDMol()
        try:
            target_o_idx = random.choice(self._target_atoms)
            rw_mol = Chem.RWMol(rdkit_mol)
            
            c_methyl_idx = rw_mol.AddAtom(Chem.Atom(6))
            rw_mol.AddBond(target_o_idx, c_methyl_idx, Chem.BondType.SINGLE)
            
            new_mol = rw_mol.GetMol()
            
            for idx in [target_o_idx, c_methyl_idx]:
                atom = new_mol.GetAtomWithIdx(idx)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                atom.SetFormalCharge(0)
                
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
            
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name

  
class NMethylation(MorphingOperator):
    def __init__(self):
        super(NMethylation, self).__init__()
        self._name = "N-methylation (Primary Amines)"
        self._target_atoms = []
        
        # Ενισχυμένο SMARTS Pattern για Πρωτοταγείς Αμίνες:
        # Ζητάμε άζωτο με 2 υδρογόνα [N;H2]
        # Αποκλείουμε: Αμίδια (!$(N-C=O)), Υδραζίνες (!$(NN)), 
        # Σουλφοναμίδια (!$(N-S(=O)=O)), Ουρίες/Καρβαμιδικά (!$(N-C(=O)))
        self.PATTERN = Chem.MolFromSmarts("[N;H2;!$(N-C=O);!$(N-S(=O)=O);!$(NN)]")

    def setOriginal(self, mol):
        super(NMethylation, self).setOriginal(mol)
        self._target_atoms = []
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return
        
        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            self._target_atoms.append(match[0])

    def morph(self):
        if not self.original or not self._target_atoms:
            return MolpherMol(other=self.original.asRDMol())
        
        rdkit_mol = self.original.asRDMol()
        try:
            target_n_idx = random.choice(self._target_atoms)
            rw_mol = Chem.RWMol(rdkit_mol)
            
            # Προσθήκη του Άνθρακα του Μεθυλίου (-CH3)
            c_methyl_idx = rw_mol.AddAtom(Chem.Atom(6))
            rw_mol.AddBond(target_n_idx, c_methyl_idx, Chem.BondType.SINGLE)
            
            new_mol = rw_mol.GetMol()
            
            # Reset ιδιοτήτων στα άτομα που τροποποιήθηκαν
            for idx in [target_n_idx, c_methyl_idx]:
                atom = new_mol.GetAtomWithIdx(idx)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                atom.SetFormalCharge(0)
                
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
            
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name
    

class SMethylation(MorphingOperator):
    def __init__(self):
        super(SMethylation, self).__init__()
        self._name = "S-methylation (Thiols)"
        self._target_atoms = []
        self.PATTERN = Chem.MolFromSmarts("[SX2H][#6;!$(C=O);!$(C=S)]")

    def setOriginal(self, mol):
        super(SMethylation, self).setOriginal(mol)
        self._target_atoms = []
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return
        
        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            # Κρατάμε το index του θείου [-SH] (το πρώτο άτομο του match, index 0)
            self._target_atoms.append(match[0])

    def morph(self):
        if not self.original or not self._target_atoms:
            return MolpherMol(other=self.original.asRDMol())
        
        rdkit_mol = self.original.asRDMol()
        try:
            target_s_idx = random.choice(self._target_atoms)
            rw_mol = Chem.RWMol(rdkit_mol)
            
            c_methyl_idx = rw_mol.AddAtom(Chem.Atom(6))
            rw_mol.AddBond(target_s_idx, c_methyl_idx, Chem.BondType.SINGLE)
            
            new_mol = rw_mol.GetMol()
            
            for idx in [target_s_idx, c_methyl_idx]:
                atom = new_mol.GetAtomWithIdx(idx)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                atom.SetFormalCharge(0)
                
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
            
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name

    
oxidize_alcohol_op = OxidizeAlcohol()
oxidize_aldehyde_op = OxidizeAldehydeToAcid() 
op_hydration = AlkeneToAlcohol()
op_dehydration = AlcoholToAlkene()
hetero_op = HeteroatomOxidation()
hydrolize_ester = HydrolyzeEster()
dealk_op = NDealkylation()
hydroxylation_op = AromaticHydroxylation()
epox_op = AliphaticEpoxidation()
azo_op = AzoReduction()
nitro_op = NitroReduction()
aldehyde_op = AldehydeReduction()
ketone_op = KetoneReduction()
alkene_op = AlkenylReduction()
lactam_op = LactamFormation()
acyl_glucuronidation = AcylGlucuronidation()
alph_glucoronidation = AlcoholPhenolGlucuronidation()
sulfation_op = AlcoholPhenolSulfation()  
nitrogen_sulfation = NitrogenSulfation()
amino_op = AminoAcidConjugation()
nacet_op = NAcetylation()
hydrazine_acetylation_op = HydrazineAcetylation()  
hydrazide_acetylation_op = HydrazideAcetylation()
omethylation_op = OMethylation()  
nmethylation_op = NMethylation()
smethylation_op = SMethylation()  


start_smiles = input("Δώσε SMILES για το start molecule: ")
target_smiles = input("Δώσε SMILES για το target molecule: ")
start_mol = MolpherMol(start_smiles)
target_mol = MolpherMol(target_smiles)
rdkit_start = start_mol.asRDMol()

selected_operators = []

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[OX2H][#6X4;H1,H2]")):
    selected_operators.append(oxidize_alcohol_op)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[CX3H1](=O)[#6,#1]")):
    selected_operators.append(oxidize_aldehyde_op)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[CX3;H1,H2]=[CX3;H0,H1,H2]")):
    selected_operators.append(op_hydration)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[OX2H][#6X4;H1,H2;!$(C(O)=O)]")):
    selected_operators.append(op_dehydration)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[N;X3;H0;!$(N-C=O);!a](-[#6])-[#6]")):
    selected_operators.append(hetero_op)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[S;X2;H0;!a]")):
    selected_operators.append(hetero_op)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[S;X3;D3;H0](=O)")):
    selected_operators.append(hetero_op)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[CX3](=O)[OX2][#6]")):
    selected_operators.append(hydrolize_ester)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[NX3;H0,H1;!a;!$(N-C=O)][CX4;H1,H2,H3]")):
    selected_operators.append(dealk_op)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[c;H1]")):
    selected_operators.append(hydroxylation_op)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[CX3;!a]=[CX3;!a]")):
    selected_operators.append(epox_op)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[C,c]-[N;!R]=[N;!R]-[C,c]")):
    selected_operators.append(azo_op)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[C,c][N;X3](=[O,O-])~[O,O-]")):
    selected_operators.append(nitro_op)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[CX3H1;!$(C(=O)[O,N,S,F,Cl,Br,I])]=[OX1]")):
    selected_operators.append(aldehyde_op)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[CX3;$(C(=O)(-[#6])-[#6])]=O")):
    selected_operators.append(ketone_op)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[CX3;!a]=[CX3;!a]")):
    selected_operators.append(alkene_op)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[NX3;R;!$(N-C=O);!a]-[CX4;R;H2;!$(C-O)]")):
    selected_operators.append(lactam_op)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[CX3](=O)[OX2H]")):
    selected_operators.append(acyl_glucuronidation)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[#6;!$(C=O);!$(C=C)][OX2H]")):
    selected_operators.append(alph_glucoronidation)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[OX2H][#6;!$(C=O)]")):
    selected_operators.append(sulfation_op)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[N;X3;!$(N-C=O);!$(N-C(=O)N);!$(N-C(=O)O)]")):
    selected_operators.append(nitrogen_sulfation)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[CX3](=O)[OX2H]")):
    selected_operators.append(amino_op)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[N;H2;!$(N-C=O);!$(N-S(=O)=O);!$(NN)]")):
    selected_operators.append(nacet_op)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[NX3;!$(N-C=O)][NX3;H2;!$(N-C=O)]")):
    selected_operators.append(hydrazine_acetylation_op)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[CX3](=O)[NX3;H1][NX3;H2;!$(N-C=O)]")):
    selected_operators.append(hydrazide_acetylation_op)

if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[OX2H][#6;!$(C=O);!$(C=C);!$(C=N)]")):
    selected_operators.append(omethylation_op)
    
if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[N;H2;!$(N-C=O);!$(N-S(=O)=O);!$(NN)]")):
    selected_operators.append(nmethylation_op)
    
if rdkit_start.HasSubstructMatch(Chem.MolFromSmarts("[SX2H][#6;!$(C=O);!$(C=S)]")):
    selected_operators.append(smethylation_op)

tree = ETree.create(source=start_mol, target=target_mol)
tree.morphing_operators = tuple(dict.fromkeys(selected_operators))

class FindClosest:
    def __init__(self):
        self.closest_mol = None
        self.closest_distance = None
    def __call__(self, morph):
        if not self.closest_mol or self.closest_distance > morph.dist_to_target:
            self.closest_mol = morph
            self.closest_distance = morph.dist_to_target

closest_info = FindClosest()

while not tree.path_found:
    tree.generateMorphs()
    tree.sortMorphs()
    tree.filterMorphs()
    tree.extend()
    tree.prune()
    tree.traverse(closest_info)
        
    print(f"Generation #{tree.generation_count}")
    print(f"Molecules in tree: {tree.mol_count}")
    print(f"Closest to target: {closest_info.closest_mol.getSMILES()} (Distance: {closest_info.closest_distance:.4f})")
    print("-" * 40)
        
    if tree.path_found or tree.generation_count >= 5:
        break

print("\nSearch finished!")
if tree.path_found:
    print("SUCCESS.")


Δώσε SMILES για το start molecule: CN(C)C[C@H]1CCCC[C@@]1(c1cccc(OC)c1)O
Δώσε SMILES για το target molecule: COC1=CC(C2(O)CCCCC2CN(C)S(=O)(=O)O)=CC=C1
Generation #1
Molecules in tree: 10
Closest to target: COC1=CC(C2(O)CCCCC2C[N+](C)(C)S(=O)(=O)O)=CC=C1 (Distance: 0.3036)
----------------------------------------
Generation #2
Molecules in tree: 51
Closest to target: COC1=CC(C2(O)CCCCC2CN(C)S(=O)(=O)O)=CC=C1 (Distance: 0.0000)
----------------------------------------

Search finished!
SUCCESS.
